### Introducción

Esta segunda parte del proyecto tiene como objetivo analizar los microdatos y tabulados de la ENIF 2024 del INEGI para comprender mejor el acceso al crédito en México. A partir de esta información, se estudian las diferencias sociodemográficas relacionadas con el crédito de nómina y el crédito personal, considerando variables como edad, sexo, nivel educativo, condición laboral, ingreso y entidad federativa. Este análisis permite identificar patrones de uso, barreras de acceso y características de los distintos perfiles financieros dentro de la población mexicana.

###Metodología
En la elaboración de este análisis, se utilizó una metodología SCRUM para la realización del proyecto. En la que las tareas se asignaron de la siguiente manera:

| Tarea                 | Subtarea                                                                  | Encargado  | Fecha de finalización |
|-----------------------|---------------------------------------------------------------------------|------------|------------------------|
| Descarga de datasets  |                                                                           | Sebastián  | 15/11/25               |
| Importación de datasets |                                                                         | Sebastián  | 15/11/25               |
| Lectura de datasets   |                                                                           | Sebastián  | 15/11/25               |
| Limpieza de datos     | Descartar columnas innecesarias                                           | Ambos      | 15/11/25               |
| Limpieza de datos     | Tratar datos faltantes                                                    | Ambos      | 15/11/25               |
| Limpieza de datos     | Detectar inconsistencia en los datos                                      | Ambos      | 15/11/25               |
| Limpieza de datos     | Cambiar tipos de datos                                                    | Ambos      | 15/11/25               |
| Análisis de datos     | Grupos sociodemográficos con mayor probabilidad de crédito de nómina      | Jennifer   | 19/11/25               |
| Análisis de datos     | Brecha de acceso financiero entre población indígena y no indígena        | Sebastián  | 19/11/25               |
| Análisis de datos     | Perfil de cliente con alto potencial de crédito personal                  | Jennifer   | 19/11/25               |
| Análisis de datos     | Indicadores sintéticos de acceso financiero (indígenas vs no indígenas)   | Sebastián  | 19/11/25               |
| Análisis de datos     | Características de personas con tarjeta clonada                           | Jennifer   | 19/11/25               |
| Análisis de datos     | Patrones entre compradores y no compradores de criptomonedas              | Sebastián  | 19/11/25               |


In [ ]:
pip install ftfy pyarrow

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re #Sirve para trabajar con expresiones regulares.
from collections import defaultdict #Diccionario que crea automáticamente un valor por defecto
from typing import Dict #Se usa para anotar el tipo de un diccionario en Python, especificando qué tipos de datos tendrán sus claves y valores
from typing import Set #Se usa para anotar el tipo de un conjuntos en Python, especificando qué tipos de datos tendrán sus valores
import plotly.express as px #Facilita hacer gráficos interactivos
import plotly.graph_objects as go #Para gráficos más personalizados e interactivos
from plotly.subplots import make_subplots
from ftfy import fix_text #Lbrería que sirve para arreglar textos mal codificados (problemas Unicode)
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow as pa
import pyarrow.parquet as pq
import warnings
warnings.simplefilter('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
ruta_archivo_datos = '/content/drive/MyDrive/Proyecto_final_DS/Datasets proyecto 2/conjunto_de_datos_tmodulo_enif2024.csv'

In [ ]:
def leer_catalogo(catalog_folder, filename, encoding='utf-8'):
    catalog_path = os.path.join(catalog_folder, filename)

    try:
        df_catalog = pd.read_csv(
            catalog_path,
            encoding=encoding,
            engine="pyarrow"
        )

        if len(df_catalog.columns) == 1:
            df_catalog = pd.read_csv(
                catalog_path,
                encoding=encoding,
                sep=';',
                engine="pyarrow"
            )

        df_catalog.columns = df_catalog.columns.str.lower().str.strip()
        cols = list(df_catalog.columns)

        if 'cve_ent' in cols and 'cve_mun' in cols and 'descrip' in cols:
            df_catalog['cve_completa'] = (
                df_catalog['cve_ent'].astype(str) + df_catalog['cve_mun'].astype(str)
            )
            return pd.Series(
                df_catalog.descrip.values,
                index=df_catalog.cve_completa
            ).to_dict()

        elif 'cve' in cols and 'descrip' in cols:
            return pd.Series(
                df_catalog.descrip.values,
                index=df_catalog.cve
            ).to_dict()

        else:
            raise ValueError(f"Columnas inesperadas: {cols}")

    except Exception as e:
        raise Exception(f"Error procesando catálogo {filename}: {e}")


def aplicar_mapeo(df, mapeos):
    for col, mapping in mapeos.items():
        if col in df.columns:
            df[col] = df[col].replace(mapping)
    return df


# --- LECTURA PRINCIPAL CON PYARROW ---
df_General = pd.read_csv(
    file_path,
    encoding=data_encoding,
    engine="pyarrow"
)
df_General.columns = df_General.columns.str.lower().str.strip()

mapeos = {}
for filename in os.listdir(catalog_folder):
    if filename.endswith('.csv'):
        col = filename.split('.')[0]
        mapeos[col] = leer_catalogo(catalog_folder, filename, encoding=catalog_encoding)

df_General = aplicar_mapeo(df_General, mapeos)

## Descripción de Variables Seleccionadas

| Código de Columna | Pregunta de la Encuesta |
| :---: | :--- |
| **LLAVEMOD** | Llave de identificación de la persona elegida |
| **EDAD_V** | Edad verificada de la persona elegida |
| **NIV** | 3.1 ¿Hasta qué año o grado aprobó usted en la escuela? |
| **sexo** | 3.1 ¿Sexo? |
| **GRA** | 3.1 ¿Hasta qué año o grado aprobó usted en la escuela? |
| **P3_2** | 3.2 Estado civil|
| **P3_6** | 3.6 Por sus costumbres y tradiciones, ¿usted se considera indígena? |
| **P3_8** | 3.8 Durante el mes pasado, ¿usted... |
| **P3_10** | 3.10 En su trabajo, actividad o negocio, ¿usted es... |
| **P3_11A** | 3.11a ¿Cuánto gana o recibe usted por trabajar (su actividad)? |
| **P3_11B** | 3.11b ¿Cada cuándo? |
| **P3_12** | 3.12 ¿Este ingreso es... |
| **P3_13** | 3.13 Por parte de su trabajo, ¿usted tiene derecho a los servicios médicos... |
| **P3_15** | 3.15 Hace cinco años, en junio de 2019, usted ¿en qué municipio o estado de la República vivía? |
| **P4_1** | 4.1 ¿Usted lleva un presupuesto o un registro de sus ingresos y gastos? |
| **P4_2_2** | 4.2 Para usted o su hogar ¿mantiene separado el dinero para pagos o deudas, del dinero del gasto diario? |
| **P4_2_3** | 4.2 Para usted o su hogar ¿lleva un registro de los recibos o deudas pendientes para asegurarse de no olvidar pagarlos? |
| **P4_3** | 4.3 En los últimos 12 meses, de junio de 2023 a la fecha, ¿lo que ganó o recibió cada mes fue suficiente para cubrir sus gastos (como vivienda, electricidad, agua, salud, alimentos, ropa o transporte)? |
| **P4_4_1** | 4.4 La última vez que no pudo cubrir sus gastos, ¿usted pidió prestado a familiares o personas conocidas? |
| **P4_4_2** | 4.4 La última vez que no pudo cubrir sus gastos, ¿usted utilizó el dinero que tenía ahorrado? |
| **P4_4_3** | 4.4 La última vez que no pudo cubrir sus gastos, ¿usted redujo sus gastos? |
| **P4_4_4** | 4.4 La última vez que no pudo cubrir sus gastos, ¿usted vendió o empeñó algún bien? |
| **P4_4_5** | 4.4 La última vez que no pudo cubrir sus gastos, ¿usted solicitó un adelanto salarial, trabajó horas extras o hizo trabajo temporal? |
| **P4_4_6** | 4.4 La última vez que no pudo cubrir sus gastos, ¿usted utilizó su tarjeta de crédito o solicitó un crédito en un banco o institución financiera? |
| **P4_4_7** | 4.4 La última vez que no pudo cubrir sus gastos, ¿usted se atrasó en el pago de algún crédito o préstamo? |
| **P4_5** | 4.5 ¿Usted ha tomado algún curso sobre cómo ahorrar, cómo hacer un presupuesto o sobre el uso responsable del crédito? |
| **P4_6_2** | 4.6.2 Le voy a hacer unas preguntas. Use las respuestas que vienen en la tarjeta (ENTREGUE LA TARJETA 2). Generalmente, ¿paga sus cuentas a tiempo (tarjeta de crédito, servicios, crédito, etcétera)? |
| **P4_6_4** | 4.6.4 Le voy a hacer unas preguntas. Use las respuestas que vienen en la tarjeta (ENTREGUE LA TARJETA 2). Generalmente, ¿se pone metas económicas a largo plazo y se esfuerza por alcanzarlas (comprar casa, ahorrar para el retiro, pagar vacaciones o fiestas, comenzar un negocio, etcétera)? |
| **P4_6_6** | 4.6.6 Le voy a hacer unas preguntas. Use las respuestas que vienen en la tarjeta (ENTREGUE LA TARJETA 2). Generalmente, ¿le sobra dinero a fin de mes? |
| **P4_7_1** | 4.7 De las siguientes frases, dígame si las considera verdaderas o falsas. La inflación significa que aumenta el precio de las cosas. |
| **P4_8_6** | 4.8 De las siguientes frases, usando las respuestas de la tarjeta (ENTREGUE LA TARJETA 3). Dígame si está de acuerdo o en desacuerdo. Se siente tranquila(o) de que su dinero sea suficiente |
| **P4_9_2** | 4.9 Si el día de hoy, se le presentara la oportunidad de comprar una casa, un terreno o abrir un negocio, ¿usted podría aprovecharla solicitando un crédito a un banco o institución financiera/ usando su tarjeta de crédito? |
| **P5_4_1** | 5.4 ¿Usted tiene cuenta o tarjeta de nómina (donde depositan su sueldo)? |
| **P5_5_1** | 5.5 ¿Con su cuenta o tarjeta de nómina (donde depositan su sueldo) tiene tarjeta de débito, es decir, el plástico con el que puede retirar y depositar dinero? |
| **P5_6_1** | 5.6 De junio de 2023 a la fecha, ¿usted guardó o ahorró en su cuenta o tarjeta de nómina (donde depositan su sueldo)? |
| **FILTRO_S5_2** | FILTRO 2: ¿TIENE CUENTA O TARJETA DE NÓMINA (5.4.1 = 1)? |
| **P5_19** | 5.19 ¿Alguna vez tuvo una cuenta o tarjeta de un banco, institución financiera o de apoyo de gobierno? |
| **P5_20** | 5.20 ¿Cuál es la razón principal por la que no tiene una cuenta o tarjeta? |
| **P5_21** | 5.21 ¿Cuál es la razón principal por la que dejó de tener su cuenta o tarjeta? |
| **P6_2_1** | 6.2 ¿Usted tiene tarjeta de crédito departamental o de tienda de autoservicio? |
| **P6_3_1** | 6.3 De junio de 2023 a la fecha, ¿se atrasó en el pago de su tarjeta de crédito departamental o de tienda de autoservicio? |
| **P6_4_1** | 6.4 De junio de 2023 a la fecha, ¿se atrasó en el pago de su tarjeta de crédito departamental o de tienda de autoservicio? - Sí |
| **P6_2_2** | 6.2 ¿Usted tiene tarjeta de crédito bancaria u otra institución financiera? |
| **P6_3_2** | 6.3 De junio de 2023 a la fecha, ¿se atrasó en el pago de su tarjeta de crédito bancaria u otra institución financiera? |
| **P6_4_2** | 6.4 De junio de 2023 a la fecha, ¿se atrasó en el pago de su tarjeta de crédito bancaria u otra institución financiera? - Sí |
| **P6_2_3** | 6.2 ¿Usted tiene crédito de nómina? |
| **P6_3_3** | 6.3 De junio de 2023 a la fecha, ¿se atrasó en el pago de su crédito de nómina? |
| **P6_4_3** | 6.4 De junio de 2023 a la fecha, ¿se atrasó en el pago de su crédito de nómina? - Sí |
| **P6_2_5** | 6.2 ¿Usted tiene crédito automotriz? |
| **P6_2_6** | 6.2 ¿Usted tiene crédito de vivienda como Infonavit, Fovissste, banco u otra institución? |
| **P6_2_7** | 6.2 ¿Usted tiene crédito grupal, comunal o solidario (como el de Compartamos)? |
| **P6_2_8** | 6.2 ¿Usted tiene crédito contratado por internet o aplicación como Prestadero, Doopla o YoTePresto? |
| **P6_2_9** | 6.2 ¿Usted tiene otro tipo de crédito? |
| **P6_5_1** | 6.5 Respecto al (último) crédito que contrató, para tomar su decisión, ¿usted utilizó la información de el Costo Anual Total (CAT)? |
| **P6_5_2** | 6.5 Respecto al (último) crédito que contrató, para tomar su decisión, ¿usted utilizó la información de la tasa de interés? |
| **P6_5_3** | 6.5 Respecto al (último) crédito que contrató, para tomar su decisión, ¿usted utilizó la información de la penalización por pagos tardíos? |
| **P6_5_4** | 6.5 Respecto al (último) crédito que contrató, para tomar su decisión, ¿usted utilizó la información de las comisiones (por apertura, por disposición de efectivo, anualidad, entre otros)? |
| **P6_7** | 6.7 Tomando en cuenta todas las deudas que usted tiene, ¿considera... |
| **FILTRO_S6_1** | FILTRO 1: ¿TIENE TARJETA DE CRÉDITO DEPARTAMENTAL O BANCARIA (6.2.1 = 1 o 6.2.2 = 1)? |
| **P6_8** | 6.8 Generalmente, ¿cuántas veces al mes utiliza su tarjeta de crédito bancaria o departamental (para compras o pago de servicios)? |
| **P6_9** | 6.9 ¿Cuál es la razón principal por la que no utiliza su(s) tarjeta(s) de crédito bancaria(s) o departamental(es)? |
| **P6_10** | 6.10 Antes de contratar su (último) crédito, ¿usted lo comparó con otros productos, en otros bancos o en otras instituciones financieras? |
| **P6_12** | 6.12 ¿Conoce cuál es el proceso para traspasar el saldo de su(s) crédito(s) a otro banco o institución financiera? |
| **P6_13** | 6.13 ¿Alguna vez tuvo un préstamo, crédito o tarjeta de crédito en un banco, tienda o institución financiera? |
| **P6_14** | 6.14 ¿Cuál es la razón principal por la que nunca ha tenido un préstamo, crédito o tarjeta de crédito? |
| **P6_15** | 6.15 ¿Cuál es la razón principal por la que dejó de tener un crédito o tarjeta de crédito? |
| **P6_16** | 6.16 ¿Alguna vez le han rechazado una solicitud de crédito? |
| **P6_17_1** | 6.17 ¿Cuáles son las razones por las que le negaron el crédito? Problemas con el buró de crédito |
| **P6_17_2** | 6.17 ¿Cuáles son las razones por las que le negaron el crédito? No pudo comprobar ingresos o eran insuficientes |
| **P6_17_3** | 6.17 ¿Cuáles son las razones por las que le negaron el crédito? Pedían documentos que no tenía (identificación oficial, comprobante de domicilio, entre otros) |
| **P6_17_4** | 6.17 ¿Cuáles son las razones por las que le negaron el crédito? No tenía historial crediticio |
| **P6_17_5** | 6.17 ¿Cuáles son las razones por las que le negaron el crédito? Falta de garantía, fiadora, fiador o aval |
| **P6_17_6** | 6.17 ¿Cuáles son las razones por las que le negaron el crédito? Otro |
| **P6_17_9** | 6.17 ¿Cuáles son las razones por las que le negaron el crédito? No sabe |
| **P7_1_1** | 7.1 Le voy a dar esta tarjeta (ENTREGUE LA TARJETA 5). ¿Qué forma de pago utiliza con más frecuencia cuando realiza compras de 500 pesos o menos? |
| **P7_1_2** | 7.1 Le voy a dar esta tarjeta (ENTREGUE LA TARJETA 5). ¿Qué forma de pago utiliza con más frecuencia cuando realiza compras de 501 pesos o más? |
| **P7_4** | 7.4 ¿Usted ha comprado o invertido en Criptomonedas o activos virtuales como Bitcoin (ETHER, XRP, NFT)? |
| **P10_1** | 10.1 En los últimos 12 meses, de junio de 2023 a la fecha, ¿ha utilizado alguna sucursal de un banco o institución financiera? |
| **P10_2** | 10.2 ¿Cuál es la razón principal por la que no ha utilizado una sucursal? |
| **P10_5** | 10.5 ¿Cuál es la razón principal por la que no ha utilizado los cajeros automáticos? |
| **P11_1_1** | 11.1 Si usted tuviera que solicitar los servicios de un banco o cualquier otra institución financiera, ¿considera que recibiría toda la información necesaria? |
| **P11_1_2** | 11.1 Si usted tuviera que solicitar los servicios de un banco o cualquier otra institución financiera, ¿considera que resolverían su necesidad o problema económico? |
| **P11_1_3** | 11.1 Si usted tuviera que solicitar los servicios de un banco o cualquier otra institución financiera, ¿considera que estaría seguro su dinero? |
| **P11_1_4** | 11.1 Si usted tuviera que solicitar los servicios de un banco o cualquier otra institución financiera, ¿considera que resolverían sus quejas y reclamaciones? |
| **P11_1_5** | 11.1 Si usted tuviera que solicitar los servicios de un banco o cualquier otra institución financiera, ¿considera que protegerían sus datos personales? |
| **P11_2_1** | 11.2 En los últimos tres años, de junio de 2021 a la fecha ¿le han clonado su tarjeta de débito o crédito para utilizarla sin su autorización? |
| **P13_3_1** | 13.3 ¿Compró o adquirió su(s) alguna vivienda o departamento principalmente con... |
| **P13_3_2** | 13.3 ¿Compró o adquirió su(s) automóvil, camioneta, camión, tráiler o moto principalmente con... |
| **P13_3_3** | 13.3 ¿Compró o adquirió su(s) alguna tierra de cultivo o terreno principalmente con... |

In [ ]:
mapeo_columnas = {

    #IDENTIFICACIÓN Y DEMOGRÁFICOS
    'llavemod': 'id_persona',
    'edad_v': 'edad',
    'niv': 'nivel_educativo',
    'gra': 'grado_aprobado',
    'sexo': 'sexo',

    #USO / ACCESO A SERVICIOS FINANCIEROS
    'p3_2': 'estado_civil',
    'p3_6': 'se_considera_indigena',
    'p3_7': 'tipo_identidad_indigena',
    'p3_8': 'situacion_ingreso_mes_pasado',
    'p3_9': 'actividad_laboral_mes_pasado',
    'p3_10': 'ocupacion',
    'p3_11a': 'ingreso_actividad_principal',
    'p3_11b': 'frecuencia_ingreso',
    'p3_12': 'ingreso_fijo_o_variable',
    'p3_13': 'tiene_seguro_medico_laboral',
    'p3_14': 'tiene_smartphone',
    'p3_15': 'residencia_hace_5_anios',

    #ACTITUDES Y GESTIÓN FINANCIERA
    'p4_1': 'lleva_presupuesto',
    'p4_3': 'percibe_ingreso_suficiente',
    'p4_5': 'tomo_curso_finanzas',
    'p4_6_2': 'paga_cuentas_a_tiempo',
    'p4_6_4': 'pone_metas_economicas',
    'p4_6_6': 'le_sobra_dinero_fin_mes',

    'p4_7_1': 'entiende_inflacion',
    'p4_8_6': 'se_siente_tranquilo_dinero',
    'p4_9_2': 'puede_pedir_credito',

    #CUENTAS, AHORRO Y NÓMINA
    'p5_4_1': 'tiene_cuenta_nomina',
    'p5_5_1': 'tiene_tarjeta_debito_nomina',
    'p5_6_1': 'ahorra_en_nomina',
    'filtro_s5_2': 'filtro_tiene_nomina',
    'p5_19': 'tuvo_cuenta_antes',
    'p5_20': 'razon_no_tiene_cuenta',
    'p5_21': 'razon_dejo_cuenta',

    #CRÉDITO
    'p6_2_1': 'tiene_tarjeta_credito_departamental',
    'p6_2_2': 'tiene_tarjeta_credito_bancaria',
    'p6_2_3': 'tiene_credito_nomina',
    'p6_2_4': 'tiene_credito_personal',
    'p6_2_5': 'tiene_credito_automotriz',
    'p6_2_6': 'tiene_credito_vivienda',
    'p6_2_7': 'tiene_credito_grupal',
    'p6_2_8': 'tiene_credito_app',
    'p6_2_9': 'tiene_otro_credito',

    # ATRASOS/PAGOS
    'p6_3_1': 'atraso_tarjeta_departamental',
    'p6_3_2': 'atraso_tarjeta_bancaria',
    'p6_3_3': 'atraso_credito_nomina',
    'p6_3_4': 'atraso_credito_personal',

    # Información usada
    'p6_5_1': 'uso_cat_credito',
    'p6_5_2': 'uso_tasa_interes_credito',
    'p6_5_3': 'uso_penalizacion_credito',
    'p6_5_4': 'uso_comisiones_credito',

    'p6_7': 'evaluacion_deuda_total',
    'p6_13': 'tuvo_credito_antes',
    'p6_16': 'le_rechazaron_credito',

    # PAGOS, MEDIOS DE PAGO
    'p7_1_1': 'forma_pago_menor_500',
    'p7_1_2': 'forma_pago_mayor_500',
    'p7_4': 'compro_criptomonedas',

    # USO DE CANALES FINANCIEROS
    'p10_1': 'uso_sucursal_12m',
    'p10_2': 'razon_no_usa_sucursal',
    'p10_5': 'razon_no_usa_cajero',

    # 8. CONFIANZA Y PROTECCIÓN
    'p11_1_1': 'confia_recibir_info',
    'p11_1_2': 'confia_resolver_necesidades',
    'p11_1_3': 'confia_seguridad_dinero',
    'p11_1_4': 'confia_resolver_quejas',
    'p11_1_5': 'confia_proteger_datos',

    'p11_2_1': 'le_clonaron_tarjeta',

    # 9. PROPIEDAD DE ACTIVOS
    'p13_3_1': 'medio_adquisicion_vivienda',
    'p13_3_2': 'medio_adquisicion_auto',
    'p13_3_3': 'medio_adquisicion_terreno'
}


In [ ]:
columnas_a_seleccionar = list(mapeo_columnas.keys())

try:
    df_final_analisis = df_General[columnas_a_seleccionar].copy()

    df_final_analisis.rename(columns=mapeo_columnas, inplace=True)

    print(f"Total de columnas en df_final_analisis: {len(df_final_analisis.columns)}")
    print("Nombres de las nuevas columnas:")
    print(df_final_analisis.columns.tolist())

except KeyError as e:
    print(f" Una columna clave en el mapeo no existe en df_General. Columna faltante: {e}")
except NameError:
    print("El DataFrame 'df_General' no está definido.")

Total de columnas en df_final_analisis: 68
Nombres de las nuevas columnas:
['id_persona', 'edad', 'nivel_educativo', 'grado_aprobado', 'sexo', 'estado_civil', 'se_considera_indigena', 'tipo_identidad_indigena', 'situacion_ingreso_mes_pasado', 'actividad_laboral_mes_pasado', 'ocupacion', 'ingreso_actividad_principal', 'frecuencia_ingreso', 'ingreso_fijo_o_variable', 'tiene_seguro_medico_laboral', 'tiene_smartphone', 'residencia_hace_5_anios', 'lleva_presupuesto', 'percibe_ingreso_suficiente', 'tomo_curso_finanzas', 'paga_cuentas_a_tiempo', 'pone_metas_economicas', 'le_sobra_dinero_fin_mes', 'entiende_inflacion', 'se_siente_tranquilo_dinero', 'puede_pedir_credito', 'tiene_cuenta_nomina', 'tiene_tarjeta_debito_nomina', 'ahorra_en_nomina', 'filtro_tiene_nomina', 'tuvo_cuenta_antes', 'razon_no_tiene_cuenta', 'razon_dejo_cuenta', 'tiene_tarjeta_credito_departamental', 'tiene_tarjeta_credito_bancaria', 'tiene_credito_nomina', 'tiene_credito_personal', 'tiene_credito_automotriz', 'tiene_credit

In [ ]:
df_final_analisis.head().T

,0,1,2,3,4
id_persona,101101,210101,303102,401101,503101
edad,38,36,20,59,37
nivel_educativo,Licenciatura o ingenierÃ­a (profesional),Licenciatura o ingenierÃ­a (profesional),Secundaria,Secundaria,Preparatoria o bachillerato
grado_aprobado,4,5,3,3,3
sexo,Hombre,Hombre,Hombre,Hombre,Mujer
estado_civil,EstÃ¡ casada(o),EstÃ¡ separada(o),Es soltera(o),Es viuda(o),EstÃ¡ casada(o)
se_considera_indigena,No,SÃ­,No,No,No
tipo_identidad_indigena,NaN,Porque naciÃ³ o pertenece a una comunidad indÃ...,NaN,NaN,NaN
situacion_ingreso_mes_pasado,TrabajÃ³ por lo menos una hora,TrabajÃ³ por lo menos una hora,TrabajÃ³ por lo menos una hora,TrabajÃ³ por lo menos una hora,TrabajÃ³ por lo menos una hora
actividad_laboral_mes_pasado,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df_final_analisis.head())

   id_persona  edad                           nivel_educativo  grado_aprobado  \
0      101101    38  Licenciatura o ingenierÃ­a (profesional)               4   
1      210101    36  Licenciatura o ingenierÃ­a (profesional)               5   
2      303102    20                                Secundaria               3   
3      401101    59                                Secundaria               3   
4      503101    37               Preparatoria o bachillerato               3   

     sexo       estado_civil se_considera_indigena  \
0  Hombre    EstÃ¡ casada(o)                    No   
1  Hombre  EstÃ¡ separada(o)                   SÃ­   
2  Hombre      Es soltera(o)                    No   
3  Hombre        Es viuda(o)                    No   
4   Mujer    EstÃ¡ casada(o)                    No   

                             tipo_identidad_indigena  \
0                                                NaN   
1  Porque naciÃ³ o pertenece a una comunidad indÃ...   
2                   

In [ ]:
conteo_nulos = df_final_analisis.isnull().sum()

total_filas = len(df_final_analisis)
porcentaje_nulos = (conteo_nulos / total_filas) * 100

df_nulos = pd.DataFrame({
    'Total Nulos': conteo_nulos,
    'Porcentaje Nulos': porcentaje_nulos.round(2).astype(str) + '%'
})
pd.set_option('display.max_rows', None)
df_nulos_ordenado = df_nulos.sort_values(by='Total Nulos', ascending=False)

print("Conteo y Porcentaje de Valores Nulos para TODAS las columnas:")
df_nulos_ordenado


Conteo y Porcentaje de Valores Nulos para TODAS las columnas:


,Total Nulos,Porcentaje Nulos
atraso_credito_nomina,13024,96.46%
atraso_credito_personal,12904,95.57%
razon_dejo_cuenta,12126,89.81%
medio_adquisicion_terreno,12060,89.32%
atraso_tarjeta_bancaria,11306,83.74%
razon_no_tiene_cuenta,10532,78.0%
atraso_tarjeta_departamental,10392,76.97%
tipo_identidad_indigena,9697,71.82%
medio_adquisicion_auto,9264,68.61%
tiene_tarjeta_debito_nomina,9197,68.12%


In [ ]:
len(df_final_analisis.columns)

68

In [ ]:
df_final_analisis.duplicated().sum()

np.int64(0)

In [ ]:
df = df_final_analisis.loc[:, df_final_analisis.isnull().mean() < 0.40]
len(df.columns)

47

In [ ]:
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(lambda x: fix_text(fix_text(str(x))))


In [ ]:
df.sample(n=10)

,id_persona,edad,nivel_educativo,grado_aprobado,sexo,estado_civil,se_considera_indigena,situacion_ingreso_mes_pasado,ocupacion,ingreso_actividad_principal,...,forma_pago_menor_500,forma_pago_mayor_500,compro_criptomonedas,uso_sucursal_12m,confia_recibir_info,confia_resolver_necesidades,confia_seguridad_dinero,confia_resolver_quejas,confia_proteger_datos,le_clonaron_tarjeta
2007,197614101,45,Estudios técnicos con preparatoria terminada,1,Mujer,Es soltera(o),No,Trabajó por lo menos una hora,Empleada(o) u obrera(o),1800.0,...,Efectivo,Efectivo,No,No,Sí,Sí,Sí,Sí,Sí,No
3247,319802101,44,Licenciatura o ingeniería (profesional),4,Mujer,Está casada(o),No,Trabajó por lo menos una hora,Empleada(o) u obrera(o),16000.0,...,Efectivo,Transferencia electrónica o aplicación de celular,No,No,Sí,Sí,Sí,Sí,Sí,No
2934,288904101,77,Primaria,6,Mujer,Está casada(o),No,Se dedica a los quehaceres del hogar o al cuid...,nan,NaN,...,Efectivo,Efectivo,No,No,No sabe,No,Sí,No,Sí,No
6715,662503101,72,Primaria,6,Mujer,Es viuda(o),Sí,Es persona jubilada o pensionada,nan,NaN,...,Efectivo,Efectivo,No,Sí,Sí,No sabe,Sí,No sabe,Sí,No
926,90903101,64,Primaria,6,Hombre,Vive con su pareja en unión libre,No,Es persona jubilada o pensionada,nan,NaN,...,Efectivo,Efectivo,No,Sí,Sí,Sí,No,No,No,No
7125,702809101,29,Licenciatura o ingeniería (profesional),5,Hombre,Vive con su pareja en unión libre,Sí,Trabajó por lo menos una hora,Empleada(o) u obrera(o),3500.0,...,Efectivo,Uso físico de tarjeta de débito o crédito,No,No,Sí,Sí,Sí,Sí,Sí,No
2560,251705103,25,Primaria,6,Hombre,Está separada(o),No,Trabajó por lo menos una hora,Empleada(o) u obrera(o),1700.0,...,Efectivo,Efectivo,No,Sí,No sabe,No,No,No,No,No
9818,967804102,58,Estudios técnicos con secundaria terminada,3,Hombre,Vive con su pareja en unión libre,No,Trabajó por lo menos una hora,Empleada(o) u obrera(o),2500.0,...,Efectivo,Efectivo,No,Sí,Sí,Sí,Sí,No sabe,No sabe,No
10096,995513104,34,Primaria,6,Mujer,Vive con su pareja en unión libre,Sí,Trabajó por lo menos una hora,Ayudante con pago,800.0,...,Efectivo,Efectivo,No,No,Sí,Sí,Sí,Sí,Sí,No
2028,199701101,51,Preparatoria o bachillerato,3,Hombre,Está separada(o),No,Trabajó por lo menos una hora,Empleada(o) u obrera(o),10000.0,...,Efectivo,Efectivo,No,No,No,No,No,No,No,Sí


### Análisis de datos

### Grupos sociodemográficos con mayor probabilidad de crédito de nómina

In [ ]:
df['edad_rango'] = pd.cut(df['edad'], bins=[0, 24, 34, 44, 54, 64, 120],
                          labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+'], include_lowest=True)

df['ingreso_rango'] = pd.cut(df['ingreso_actividad_principal'], bins=[0, 3000, 7000, 15000, 30000, 60000, np.inf],
                             labels=['0-3k','3k-7k', '7k-15k', '15k-30k', '30k-60k', '60k+'], include_lowest=True)

#Función para binarizar columnas
def binarizar(df, columnas):
    mapa = {
        'Sí': 1, 'Si': 1, 'SÍ': 1, 'SI': 1, 'sí': 1, 'si': 1,
        'No': 0, 'NO': 0, 'no': 0,
        'SA': None, 'sa': None,
        'Ns/Nc': None, 'NS/NC': None, 'ns/nc': None
    }

    mapa_frecuencia = {
        'Siempre': 1,
        'Algunas veces': 0,
        'Nunca': 0,
        'No sabe': None,
        'No responde': None
    }

    mapa_verdadero_falso = {
        'Verdadera': 1, 'Verdadero': 1, 'verdadera': 1, 'verdadero': 1,
        'Falsa': 0, 'Falso': 0, 'falsa': 0, 'falso': 0,
        'No sabe': None,
        'No responde': None
    }

    df_copy = df.copy()

    for col in columnas:
        if col not in df_copy.columns:
            continue

        df_copy[col] = df_copy[col].astype(str).str.strip()

        if col in ['paga_cuentas_a_tiempo', 'pone_metas_economicas', 'le_sobra_dinero_fin_mes']:
            mapeo = mapa_frecuencia
        elif col in ['entiende_inflacion']:
            mapeo = mapa_verdadero_falso
        else:
            mapeo = mapa

        df_copy[col] = df_copy[col].replace(mapeo)
        df_copy[col] = df_copy[col].fillna(0).astype('int8')

    return df_copy


#Función que calcula la tasa de crédito por categoría
def tasa_credito(df, columna, objetivo="tiene_credito_nomina"):
    if columna not in df.columns:
        return None

    df_temporal = df[[columna, objetivo]].dropna()

    tabla = (
        df_temporal.groupby(columna)[objetivo]
        .mean()
        .reset_index()
    )

    tabla["probabilidad_credito_nomina"] = tabla[objetivo] * 100
    tabla = tabla.drop(columns=[objetivo])

    return tabla.sort_values("probabilidad_credito_nomina", ascending=False)


def analizar_credito_nomina(df):
    variables = [
        "edad_rango",
        "sexo",
        "nivel_educativo",
        "ocupacion",
        "ingreso_rango",
        "ingreso_fijo_o_variable",
        "frecuencia_ingreso",
        "estado_civil"
    ]

    resultados = {}

    for var in variables:
        tabla = tasa_credito(df, var)
        if tabla is not None:
            resultados[var] = tabla

    return resultados

df_limp = binarizar(df, ["tiene_credito_nomina"])

resultados_analisis_credito_nomina = analizar_credito_nomina(df_limp)

for variable, tabla in resultados_analisis_credito_nomina.items():
    print(f"Probabilidad de tener crédito de nómina según: {variable}")
    print(display(tabla.head()))

Probabilidad de tener crédito de nómina según: edad_rango


,edad_rango,probabilidad_credito_nomina
2,35-44,4.679626
1,25-34,4.462990
3,45-54,4.351430
4,55-64,3.210273
0,18-24,2.247874


None
Probabilidad de tener crédito de nómina según: sexo


,sexo,probabilidad_credito_nomina
0,Hombre,4.439329
1,Mujer,2.803235


None
Probabilidad de tener crédito de nómina según: nivel_educativo


...

None
Probabilidad de tener crédito de nómina según: frecuencia_ingreso


,frecuencia_ingreso,probabilidad_credito_nomina
0,A la quincena,10.606061
3,Al mes,7.072600
1,A la semana,2.919574
4,nan,0.892478
2,Al año,0.000000


None
Probabilidad de tener crédito de nómina según: estado_civil


,estado_civil,probabilidad_credito_nomina
3,Está divorciada(o),6.496063
5,Vive con su pareja en unión libre,3.763040
2,Está casada(o),3.728671
4,Está separada(o),3.681443
0,Es soltera(o),3.291787


None


In [ ]:
def quitar_parentesis_largos(texto):
    #Quita cualquier (contenido) donde el contenido tiene más de 5 letras
    return re.sub(r'\([A-Za-zÁÉÍÓÚáéíóúñÑ]{6,}\)', '', texto).strip()

def graficar_tasas_subplots(resultados, columnas_por_fila=2):
    variables = list(resultados.keys())
    n = len(variables)

    filas = (n + columnas_por_fila - 1) // columnas_por_fila

    # Paleta de colores moderna y vibrante
    colores = [
        '#667eea',
        '#f093fb',
        '#4facfe',
        '#43e97b',
        '#fa709a',
        '#fee140',
        '#30cfd0',
        '#c471ed'
    ]

    subplot_titles = [
        f"<b>{quitar_parentesis_largos(var.replace('_', ' ').title())}</b>"
        for var in variables
    ]


    fig = make_subplots(
        rows=filas,
        cols=columnas_por_fila,
        subplot_titles=[f"<b>{var.replace('_', ' ').title()}</b>" for var in variables],
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )

    fila = 1
    col = 1

    for idx, (variable, tabla) in enumerate(resultados.items()):
        tabla = tabla.rename(columns={"tasa_credito_nomina": "probabilidad_credito_nomina"})

        #Limpiar texto entre parentesis en la primera columna
        #Solo quita parentesis con más de 2 letras
        tabla_limpia = tabla.copy()
        primera_col = tabla_limpia.columns[0]

        def limpiar_parentesis(texto):
            #Busca parentesis y verifica si tiene más de 2 letras
            def verificar_contenido(match):
                contenido = match.group(1)
                #Contar solo letras (no espacios ni otros caracteres)
                letras = re.findall(r'[a-zA-ZáéíóúÁÉÍÓÚñÑ]', contenido)
                if len(letras) > 2:
                    return ''  #Quitar el parentesis completo
                else:
                    return match.group(0)  #Mantener el paréntesis

            return re.sub(r'\s*\(([^)]*)\)', verificar_contenido, texto).strip()

        tabla_limpia[primera_col] = tabla_limpia[primera_col].astype(str).apply(limpiar_parentesis)

        # Seleccionar color
        color_base = colores[idx % len(colores)]

        # Obtener valores
        valores = tabla["probabilidad_credito_nomina"]

        # Detectar si los valores ya están en porcentaje (>1) o son decimales (<1)
        valores_normalizados = valores / 100 if valores.max() > 1 else valores

        # Colores individuales por barra
        colores_barras = [color_base for _ in range(len(valores))]

        fig.add_trace(
            go.Bar(
                x=tabla_limpia.iloc[:, 0],
                y=valores_normalizados,
                marker=dict(
                    color=colores_barras,
                    line=dict(color='rgba(255,255,255,0.3)', width=1.5),
                    opacity=0.85
                ),
                hovertemplate='<b>%{x}</b><br>Probabilidad: %{y:.1%}<extra></extra>',
                text=[f'{v:.1%}' for v in valores_normalizados],
                textposition='outside',
                textfont=dict(size=10, color='#2d3748')
            ),
            row=fila,
            col=col
        )

        # Formato del eje Y
        fig.update_yaxes(
            title_text="Probabilidad",
            tickformat='.0%',
            showgrid=True,
            gridcolor='rgba(200,200,200,0.2)',
            row=fila,
            col=col
        )

        # Formato del eje X
        fig.update_xaxes(
            showgrid=False,
            row=fila,
            col=col
        )

        col += 1
        if col > columnas_por_fila:
            col = 1
            fila += 1

    fig.update_layout(
        height=350 * filas,
        showlegend=False,
        title={
            'text': "<b>Probabilidad de Crédito de Nómina por Variables Sociodemográficas</b>",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'color': '#2d3748'}
        },
        plot_bgcolor='#f7fafc',
        paper_bgcolor='white',
        font=dict(family="Arial, sans-serif", size=11, color='#4a5568'),
        margin=dict(t=100, l=60, r=60, b=60)
    )

    # Actualizar estilo de los títulos de los subplots
    for annotation in fig.layout.annotations:
        annotation.font.size = 13
        annotation.font.color = '#2d3748'

    fig.show()

df_clean = binarizar(df, ["tiene_credito_nomina"])
resultados = analizar_credito_nomina(df_clean)

graficar_tasas_subplots(resultados, columnas_por_fila=2)

**Resultado**:

El análisis muestra que el acceso al crédito de nómina esta muy ligado a la estabilidad laboral, las condiciones socieconómicas y la regularidad de los ingresos, mietras que los grupos más vulnerables o informales tienen más barreras para acceder al crédito de nómina.

Los grupos con mayor probabilidad de contar con el crédito son:

*   Hombres.
*   Tener entre 35 y 54.
*   Con alto nivel educativo(profesionales, maestros y quienes tiene posgrado).
*   Empleados formales.
*   Tienen ingreso fijo.
*   Sueldo de más de 15 mil pesos mensuales.
*   Recibe pago de manera quincenal.










### Perfil de cliente con alto potencial de crédito personal

In [ ]:
# Redefine la función binarizar localmente para incluir 'No sabe' en el mapeo
def binarizar(df, columnas):
    mapa = {
        'Sí': 1, 'Si': 1, 'SÍ': 1, 'SI': 1, 'sí': 1, 'si': 1,
        'No': 0, 'NO': 0, 'no': 0,
        'SA': None, 'sa': None,
        'Ns/Nc': None, 'NS/NC': None, 'ns/nc': None,
        'No sabe': None, 'No responde': None # Añadido 'No sabe' y 'No responde'
    }

    mapa_frecuencia = {
        'Siempre': 1,
        'Algunas veces': 0,
        'Nunca': 0,
        'No sabe': None,
        'No responde': None
    }

    mapa_verdadero_falso = {
        'Verdadera': 1, 'Verdadero': 1, 'verdadera': 1, 'verdadero': 1,
        'Falsa': 0, 'Falso': 0, 'falsa': 0, 'falso': 0,
        'No sabe': None,
        'No responde': None
    }

    df_copy = df.copy()

    for col in columnas:
        if col not in df_copy.columns:
            continue

        # Convertir a string para manejar uniformemente antes de reemplazar
        df_copy[col] = df_copy[col].astype(str).str.strip()

        if col in ['paga_cuentas_a_tiempo', 'pone_metas_economicas', 'le_sobra_dinero_fin_mes']:
            mapeo_actual = mapa_frecuencia
        elif col in ['entiende_inflacion']:
            mapeo_actual = mapa_verdadero_falso
        else:
            mapeo_actual = mapa

        # Aplicar mapeo y luego rellenar NaNs antes de convertir a int
        df_copy[col] = df_copy[col].replace(mapeo_actual)
        # Convertir 'nan' string resultante de .astype(str) a NaN numérico antes de fillna
        df_copy[col] = df_copy[col].replace('nan', np.nan)
        df_copy[col] = df_copy[col].fillna(0).astype('int8')

    return df_copy

# Binarizar ambas columnas a la vez
df_limp = binarizar(df, ["le_clonaron_tarjeta", "tiene_credito_nomina"])

# Crear grupos
grupo_clonacion = df_limp[df_limp["le_clonaron_tarjeta"] == 1]
grupo_no_clonacion = df_limp[df_limp["le_clonaron_tarjeta"] == 0]

# Calcular tasas
tasa_credito_clonados = grupo_clonacion["tiene_credito_nomina"].mean()
tasa_credito_no_clonados = grupo_no_clonacion["tiene_credito_nomina"].mean()


print(f"Tasa crédito (Clonados): {tasa_credito_clonados:.2%}")
print(f"Tasa crédito (No Clonados): {tasa_credito_no_clonados:.2%}")
diferencia = tasa_credito_clonados - tasa_credito_no_clonados
print(f"Diferencia (Clonados y No clonados) {diferencia:.2%}")

Tasa crédito (Clonados): 7.95%
Tasa crédito (No Clonados): 3.26%
Diferencia (Clonados y No clonados) 4.69%


In [ ]:
columnas_confianza = [
    'confia_recibir_info',
    'confia_resolver_necesidades',
    'confia_seguridad_dinero',
    'confia_resolver_quejas',
    'confia_proteger_datos'
]

# Binarizar todas las columnas
columnas_a_binarizar = ["le_clonaron_tarjeta"] + columnas_confianza
df_limp = binarizar(df, columnas_a_binarizar)

#Crear grupos
grupo_clonacion = df_limp[df_limp["le_clonaron_tarjeta"] == 1]
grupo_no_clonacion = df_limp[df_limp["le_clonaron_tarjeta"] == 0]

# Calcular promedios
promedios_clonados = grupo_clonacion[columnas_confianza].mean()
promedios_no_clonados = grupo_no_clonacion[columnas_confianza].mean()

print("Confianza (Clonados)")
print(promedios_clonados.apply(lambda x: f"{x:.2%}"))

print("\nConfianza (No clonados)")
print(promedios_no_clonados.apply(lambda x: f"{x:.2%}"))

print("\nDiferencia (Clonados y No clonados)")
diferencia = promedios_clonados - promedios_no_clonados
print(diferencia.apply(lambda x: f"{x:+.2%}"))

Confianza (Clonados)
confia_recibir_info            72.92%
confia_resolver_necesidades    60.37%
confia_seguridad_dinero        67.20%
confia_resolver_quejas         60.75%
confia_proteger_datos          57.14%
dtype: object

Confianza (No clonados)
confia_recibir_info            69.99%
confia_resolver_necesidades    54.82%
confia_seguridad_dinero        61.16%
confia_resolver_quejas         54.73%
confia_proteger_datos          60.31%
dtype: object

Diferencia (Clonados y No clonados)
confia_recibir_info            +2.93%
confia_resolver_necesidades    +5.56%
confia_seguridad_dinero        +6.05%
confia_resolver_quejas         +6.02%
confia_proteger_datos          -3.17%
dtype: object


In [ ]:
import numpy as np

def binarizar(df, columnas):
    mapa = {
        'Sí': 1, 'Si': 1, 'SÍ': 1, 'SI': 1, 'sí': 1, 'si': 1,
        'No': 0, 'NO': 0, 'no': 0,
        'SA': None, 'sa': None,
        'Ns/Nc': None, 'NS/NC': None, 'ns/nc': None,
        'No sabe': None, 'No responde': None
    }

    mapa_frecuencia = {
        'Siempre': 1,
        'Algunas veces': 0,
        'Nunca': 0,
        'No sabe': None,
        'No responde': None
    }

    mapa_verdadero_falso = {
        'Verdadera': 1, 'Verdadero': 1, 'verdadera': 1, 'verdadero': 1,
        'Falsa': 0, 'Falso': 0, 'falsa': 0, 'falso': 0,
        'No sabe': None,
        'No responde': None
    }

    # Specific map for 'ingreso_fijo_o_variable'
    mapa_ingreso_fijo_variable = {
        'Fijo': 1,
        'Variable': 0,
        'No sabe': None,
        'No responde': None,
        'nan': None # Handle string 'nan' after astype(str)
    }

    # Specific map for 'tiene_seguro_medico_laboral'
    mapa_seguro_medico_laboral = {
        'Del ISSSTE': 1,
        'Del seguro social (IMSS)': 1,
        'Del ISSEMYM (para empleados del gobierno del Estado de México)': 1,
        'De Pemex': 1,
        'De las fuerzas armadas o Marina': 1,
        'De alguna institución privada (seguro de gastos médicos mayores)': 1,
        'De alguna otra institución privada': 1,
        'Entonces, carece de derecho a servicios médicos': 0, # This means no insurance
        'Entonces, carece de derecho a servicios médicos por parte de su trabajo (incluye IMSS-Bienestar, antes seguro popular, instituto de salud para el bienestar)': 0, # Added this specific string
        'No': 0, # Explicit No
        'No sabe': None,
        'No responde': None,
        'nan': None,
        'Del ISSSTE estatal': 1, # Added this specific string from error
        'De PEMEX, Defensa o Marina': 1, # Added this specific string from traceback
        'De un seguro privado de gastos médicos': 1, # Added this specific string from error
        'De otra institución': 1 # Added this specific string from error
    }

    df_copy = df.copy()

    for col in columnas:
        if col not in df_copy.columns:
            continue

        df_copy[col] = df_copy[col].astype(str).str.strip()

        if col in ['paga_cuentas_a_tiempo', 'pone_metas_economicas', 'le_sobra_dinero_fin_mes']:
            mapeo_actual = mapa_frecuencia
        elif col in ['entiende_inflacion']:
            mapeo_actual = mapa_verdadero_falso
        elif col == 'ingreso_fijo_o_variable': # New condition for the problematic column
            mapeo_actual = mapa_ingreso_fijo_variable
        elif col == 'tiene_seguro_medico_laboral': # New condition for the problematic column
            mapeo_actual = mapa_seguro_medico_laboral
        else:
            mapeo_actual = mapa

        df_copy[col] = df_copy[col].replace(mapeo_actual)
        df_copy[col] = df_copy[col].replace('nan', np.nan) # Ensure 'nan' string is handled before fillna
        df_copy[col] = df_copy[col].fillna(0).astype('int8')

    return df_copy


columnas_binarias = [
    "le_clonaron_tarjeta",
    "tuvo_credito_antes",
    "ingreso_fijo_o_variable",
    "filtro_tiene_nomina",
    "tiene_seguro_medico_laboral"
]

# Binarizar
df_limp = binarizar(df, columnas_binarias)

# Crear grupos
grupo_clonados = df_limp[df_limp["le_clonaron_tarjeta"] == 1]
grupo_no_clonados = df_limp[df_limp["le_clonaron_tarjeta"] == 0]

#Comparar
columnas_para_comparar = [
    "tuvo_credito_antes",
    "ingreso_fijo_o_variable",
    "filtro_tiene_nomina",
    "tiene_seguro_medico_laboral"
]

promedio_clonados = grupo_clonados[columnas_para_comparar].mean()
promedio_no_clonados = grupo_no_clonados[columnas_para_comparar].mean()

print("Comportamiento (Clonados)")
print(promedio_clonados.apply(lambda x: f"{x:.2%}"))

print("\nComportamiento (No Clonados)")
print(promedio_no_clonados.apply(lambda x: f"{x:.2%}"))

print("\nDiferencia (Clonados - No clonados)")
print((promedio_clonados - promedio_no_clonados).apply(lambda x: f"{x:+.2%}"))

Comportamiento (Clonados)
tuvo_credito_antes             12.05%
ingreso_fijo_o_variable        41.74%
filtro_tiene_nomina            59.50%
tiene_seguro_medico_laboral    53.66%
dtype: object

Comportamiento (No Clonados)
tuvo_credito_antes             13.23%
ingreso_fijo_o_variable        29.35%
filtro_tiene_nomina            30.13%
tiene_seguro_medico_laboral    29.44%
dtype: object

Diferencia (Clonados - No clonados)
tuvo_credito_antes              -1.18%
ingreso_fijo_o_variable        +12.39%
filtro_tiene_nomina            +29.37%
tiene_seguro_medico_laboral    +24.22%
dtype: object


In [ ]:
from scipy.stats import chi2_contingency

tabla = pd.crosstab(df['le_clonaron_tarjeta'], df['tiene_credito_nomina'])
chi2, p, dof, expected = chi2_contingency(tabla)

print("p-value:", p)

p-value: 2.0917102281881385e-13


In [ ]:



#Grafica de violin
def crear_violin_plot(grupo_clonados, grupo_no_clonados):
    """
    Violin plot para ver distribución completa
    """
    # Preparar datos
    df_combined = pd.DataFrame({
        'Ingreso': pd.concat([grupo_no_clonados['ingreso_actividad_principal'],
                              grupo_clonados['ingreso_actividad_principal']]),
        'Grupo': ['No Clonados'] * len(grupo_no_clonados) + ['Clonados'] * len(grupo_clonados)
    })

    fig = go.Figure()

    # Violin para No Clonados
    fig.add_trace(go.Violin(
        y=grupo_no_clonados['ingreso_actividad_principal'],
        name='No Clonados',
        box_visible=True,
        meanline_visible=True,
        fillcolor='#66BB6A',
        opacity=0.7,
        line_color='#66BB6A',
        hovertemplate='<b>No Clonados</b><br>Ingreso: $%{y:,.0f}<extra></extra>'
    ))

    # Violin para Clonados
    fig.add_trace(go.Violin(
        y=grupo_clonados['ingreso_actividad_principal'],
        name='Clonados',
        box_visible=True,
        meanline_visible=True,
        fillcolor='#BB2124',
        opacity=0.7,
        line_color='#BB2124',
        hovertemplate='<b>Clonados</b><br>Ingreso: $%{y:,.0f}<extra></extra>'
    ))

    fig.update_layout(
        title='Distribución de Ingresos - Violin Plot',
        yaxis_title='Ingreso Mensual Aproximado ($)',
        height=600,
        template='plotly_white'
    )

    return fig

# 3. Violin Plot
print("\n3. Violin Plot")
fig3 = crear_violin_plot(grupo_clonados, grupo_no_clonados)
fig3.show()


3. Violin Plot


In [ ]:
columnas_a_binarizar = [
    "le_clonaron_tarjeta",
    "tiene_credito_nomina",
    "confia_recibir_info",
    "confia_resolver_necesidades",
    "confia_seguridad_dinero",
    "confia_resolver_quejas",
    "confia_proteger_datos",
    "tuvo_credito_antes",
    "ingreso_fijo_o_variable",
    "filtro_tiene_nomina",
    "tiene_seguro_medico_laboral"]
df_limp = binarizar(df, columnas_a_binarizar)

grupo_clonacion = df_limp[df_limp["le_clonaron_tarjeta"] == 1]
grupo_no_clonacion = df_limp[df_limp["le_clonaron_tarjeta"] == 0]

# Paleta de colores moderna
COLORS = {
    'clonados': '#E63946',
    'no_clonados': '#06A77D',
    'diferencia': '#FFB703',
    'positivo': '#06A77D',
    'negativo': '#E63946',
    'neutro': '#8338EC',
    'accent1': '#00D9FF',
    'accent2': '#FB5607',
    'accent3': '#FFBE0B'
}

def grafica_tasa_credito_nomina(grupo_clonacion, grupo_no_clonacion, COLORS):
    tasa_credito_clonados = grupo_clonacion["tiene_credito_nomina"].mean()
    tasa_credito_no_clonados = grupo_no_clonacion["tiene_credito_nomina"].mean()

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'indicator'}, {'type': 'indicator'}]],
        horizontal_spacing=0.3
    )

    # Clonados
    fig.add_trace(
        go.Indicator(
            mode='gauge+number+delta',
            value=tasa_credito_clonados * 100,
            delta={'reference': tasa_credito_no_clonados * 100, 'suffix': 'pp'},
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': COLORS['clonados'], 'thickness': 0.7},
                'bgcolor': 'rgba(230, 57, 70, 0.1)',
                'borderwidth': 2,
                'bordercolor': COLORS['clonados'],
                'steps': [
                    {'range': [0, 50], 'color': 'rgba(255,183,3,0.3)'},
                    {'range': [50, 75], 'color': 'rgba(6,168,125,0.3)'},
                    {'range': [75,100], 'color': 'rgba(230,57,70,0.3)'}
                ]
            },
            title={'text': 'Tasa Crédito<br>Clonados'},
            number={'suffix': '%', 'font': {'size': 48}}
        ),
        row=1, col=1
    )

    # No Clonados
    fig.add_trace(
        go.Indicator(
            mode='gauge+number+delta',
            value=tasa_credito_no_clonados * 100,
            delta={'reference': tasa_credito_clonados * 100, 'suffix': 'pp'},
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': COLORS['no_clonados'], 'thickness': 0.7},
                'bgcolor': 'rgba(6, 168, 125, 0.1)',
                'borderwidth': 2,
                'bordercolor': COLORS['no_clonados'],
                'steps': [
                    {'range': [0, 50], 'color': 'rgba(255,183,3,0.3)'},
                    {'range': [50, 75], 'color': 'rgba(6,168,125,0.3)'},
                    {'range': [75,100], 'color': 'rgba(230,57,70,0.3)'}
                ]
            },
            title={'text': 'Tasa Crédito<br>No Clonados'},
            number={'suffix': '%', 'font': {'size': 48}}
        ),
        row=1, col=2
    )

    fig.update_layout(
        title="Comparación de Tasa de Crédito Nómina",
        height=500,
        paper_bgcolor="white"
    )
    return fig

def grafica_frecuencia_ingresos(grupo_clonacion, grupo_no_clonacion, COLORS):
    freq_cl = grupo_clonacion["frecuencia_ingreso"].value_counts()
    freq_no = grupo_no_clonacion["frecuencia_ingreso"].value_counts()

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'pie'}, {'type': 'pie'}]],
        subplot_titles=('Clonados', 'No Clonados')
    )

    colors_pie = [
        COLORS['clonados'], COLORS['no_clonados'], COLORS['accent1'],
        COLORS['accent2'], COLORS['accent3'], COLORS['neutro']
    ]

    fig.add_trace(go.Pie(
        labels=freq_cl.index,
        values=freq_cl.values,
        marker=dict(colors=colors_pie, line=dict(color='white', width=2))
    ), row=1, col=1)

    fig.add_trace(go.Pie(
        labels=freq_no.index,
        values=freq_no.values,
        marker=dict(colors=colors_pie, line=dict(color='white', width=2))
    ), row=1, col=2)

    fig.update_layout(
        title="Distribución de Frecuencia de Ingresos",
        height=550,
        paper_bgcolor="white"
    )
    return fig
def grafica_diferencias_confianza(grupo_clonacion, grupo_no_clonacion, columnas_confianza, COLORS):
    promedios_clonados = grupo_clonacion[columnas_confianza].mean() * 100
    promedios_no_clonados = grupo_no_clonacion[columnas_confianza].mean() * 100
    diferencias_confianza = promedios_clonados - promedios_no_clonados

    fig = go.Figure()

    fig.add_trace(go.Waterfall(
        x=columnas_confianza,
        y=diferencias_confianza.values,
        connector={'line': {'color': '#2d3748'}},
        increasing=dict(marker=dict(color=COLORS['positivo'], line=dict(color='white', width=2))),
        decreasing=dict(marker=dict(color=COLORS['negativo'], line=dict(color='white', width=2))),
        text=[f'{v:+.1f}pp' for v in diferencias_confianza.values],
        textposition='outside'
    ))

    fig.add_hline(y=0, line_dash='dash', line_color='#2d3748')

    fig.update_layout(
        title="Impacto de la Clonación en Confianza",
        height=550,
        plot_bgcolor="#f7fafc",
        paper_bgcolor="white"
    )
    return fig
def grafica_ocupaciones(grupo_clonacion, grupo_no_clonacion, COLORS):
    top_cl = grupo_clonacion["ocupacion"].value_counts().head(8) * 100 / len(grupo_clonacion)
    top_no = grupo_no_clonacion["ocupacion"].value_counts().head(8) * 100 / len(grupo_no_clonacion)

    todas_ocupaciones = list(set(top_cl.index) | set(top_no.index))

    valores_cl = [top_cl.get(o, 0) for o in todas_ocupaciones]
    valores_no = [top_no.get(o, 0) for o in todas_ocupaciones]

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=valores_cl, y=todas_ocupaciones,
        name="Clonados", orientation="h",
        marker=dict(color=COLORS['clonados'], line=dict(color='white', width=2)),
        text=[f'{v:.1f}%' for v in valores_cl], textposition='outside'
    ))

    fig.add_trace(go.Bar(
        x=valores_no, y=todas_ocupaciones,
        name="No Clonados", orientation="h",
        marker=dict(color=COLORS['no_clonados'], line=dict(color='white', width=2)),
        text=[f'{v:.1f}%' for v in valores_no], textposition='outside'
    ))

    fig.update_layout(
        title="Top Ocupaciones por Grupo",
        barmode="group",
        height=600,
        paper_bgcolor="white"
    )
    return fig

graficas = [
    grafica_tasa_credito_nomina,
    grafica_diferencias_confianza,
    grafica_ocupaciones
]

for f in graficas:
    if f == grafica_diferencias_confianza:
        fig = f(grupo_clonacion, grupo_no_clonacion, columnas_confianza, COLORS)
    else:
        fig = f(grupo_clonacion, grupo_no_clonacion, COLORS)
    fig.show()



**Resultado**

Los datos muestran notorias diferencas entre personas a las que les clonaron la tarjeta y quienes no, especialmente en su nivel de integración financiera, confianza en instituciones y características laborales. La tasa de crédito de nómina es mayor entre personas que fueron víctimas de clonación (7.95%) comparada con quienres no lo fueron (3.26%). Esto significa que las personas cuya tarjeta fue clonada tiene más del doble de probabilidad de contar con crédito de nómina, lo que sugiere que hacen mayor uso del sistema financiero, poseen cuenta activas y realizan más transacciones, estos aspectos aumentan la probabilidad de ser víctima de fraude. También la prueba chi-cuadrado confirma que existe una relación entre tener nómina y haber experimientado clonación de tarjeta.

También se señala que el grupo de personas a las que les han clonado la tarjeta presentan ingresos medianos más altos. Los no clonados tienen una distribución más baja de ingresos. Lo que refuerza la hipótesis de a mayor estabilidd financiera y mayor interacción financiera, mayor riesgo de ser clonado.

Los resultados también mustran que aunque las víctimas de clonación presentan menos confianza en la protección de datos, mantienen niveles altos de confianza general.

Se observa que las personas con clonación de tarjeta se concentran más en empleos formales. Esto vuelve a conectar con el hipótesis general, el empleo formal incrementa el uso de actividades financieras y, a su vez, mayor exposición a actividades fraudulentas.

### Características y patrones en personas cuyas tarjetas han sido clonadas

In [ ]:
#Biranizar
cols_binarizar = [
    "paga_cuentas_a_tiempo",
    "lleva_presupuesto",
    "le_sobra_dinero_fin_mes",
    "entiende_inflacion",
    "pone_metas_economicas",
    "percibe_ingreso_suficiente",
    "tiene_smartphone",
    "filtro_tiene_nomina",
    "tiene_credito_personal"
]

df_bin = binarizar(df, cols_binarizar)

#Educacion alta
educacion_alta_map = {
    "Estudios técnicos con preparatoria terminada": 1,
    "Licenciatura o ingeniería (profesional)": 1,
    "Especialidad": 1,
    "Maestría": 1,
    "Doctorado": 1
}

df_bin["educacion_alta"] = (
    df_bin["nivel_educativo"]
    .map(educacion_alta_map)
    .fillna(0)
    .astype(int)
)

#Ingresos
df_bin["ingreso_actividad_principal"] = df_bin["ingreso_actividad_principal"].fillna(0)

#Percentil 60
p60 = df_bin["ingreso_actividad_principal"].quantile(0.60)
df_bin["ingreso_alto"] = (df_bin["ingreso_actividad_principal"] >= p60).astype(int)

#Puntos graduales
df_bin["puntos_ingreso"] = pd.cut(
    df_bin["ingreso_actividad_principal"],
    bins=[0, 5000, 10000, 15000, 25000, 50000, 100000],
    labels=[0, 1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

#Ingreso fijo
df_bin["ingreso_fijo"] = (
    df_bin["ingreso_fijo_o_variable"] == "Fijo"
).astype(int)

#Ingreso regular
df_bin["ingreso_regular"] = df_bin["frecuencia_ingreso"].isin(
    ["A la semana", "A la quincena", "Al mes"]
).astype(int)

#Buenos hábitos
df_bin["buen_habito"] = (
    (df_bin["paga_cuentas_a_tiempo"] == 1) &
    (df_bin["lleva_presupuesto"] == 1)
).astype(int)

#Buenos conocimientos
df_bin["buen_conocimiento"] = (
    (df_bin["entiende_inflacion"] == 1) &
    (df_bin["pone_metas_economicas"] == 1)
).astype(int)

#Estabilidad financiera
df_bin["estabilidad_financiera"] = (
    (df_bin["le_sobra_dinero_fin_mes"] == 1) &
    (df_bin["percibe_ingreso_suficiente"] == 1)
).astype(int)


#Calcular Score
df_bin["score_credito"] = (
    2.0 * df_bin["educacion_alta"] +
    2.0 * df_bin["buen_habito"] +
    2.0 * df_bin["buen_conocimiento"] +
    1.5 * df_bin["estabilidad_financiera"] +
    0.5 * df_bin["puntos_ingreso"] +
    1.0 * df_bin["ingreso_fijo"] +
    0.5 * df_bin["ingreso_regular"] +
    1.0 * df_bin["filtro_tiene_nomina"] +
    0.5 * df_bin["tiene_smartphone"]
)

#Identificar segmento potencial

umbral = df_bin["score_credito"].quantile(0.70)

df_potencial = df_bin[
    (df_bin["score_credito"] >= umbral) &
    (df_bin["tiene_credito_personal"] == 0)
].copy()

#Metricas clave
print("\nMétricas del mercado potencial")

total_personas = len(df_bin)
sin_credito = (df_bin["tiene_credito_personal"] == 0).sum()
potenciales = len(df_potencial)

print(f"\nTotal de personas: {total_personas:,}")
print(f"Sin crédito personal: {sin_credito:,}")
print(f"Potenciales (buen perfil): {potenciales:,}")
print(f"\nRepresentan el {100 * potenciales / total_personas:.2f}% del total")
print(f"Representan el {100 * potenciales / sin_credito:.2f}% de quienes no tienen crédito")
print(f"\nUmbral del score: {umbral:.2f}")

#Perfil descriptivo
print("Perfil del segmento potencia")

print(f"\nEdad:")
print(f"Promedio: {df_potencial['edad'].mean():.1f} años")
print(f"Mediana: {df_potencial['edad'].median():.0f} años")

print(f"\nIngreso:")
print(f"Promedio: ${df_potencial['ingreso_actividad_principal'].mean():,.2f}")
print(f"Mediana: ${df_potencial['ingreso_actividad_principal'].median():,.2f}")

print(f"\nCaracterísticas:")
print(f"Educación alta: {100 * df_potencial['educacion_alta'].mean():.1f}%")
print(f"Buenos hábitos: {100 * df_potencial['buen_habito'].mean():.1f}%")
print(f"Buenos conocimientos: {100 * df_potencial['buen_conocimiento'].mean():.1f}%")
print(f"Estabilidad financiera: {100 * df_potencial['estabilidad_financiera'].mean():.1f}%")
print(f"Ingreso fijo: {100 * df_potencial['ingreso_fijo'].mean():.1f}%")
print(f"Tiene nómina: {100 * df_potencial['filtro_tiene_nomina'].mean():.1f}%")

print(f"\nScore:")
print(f"Promedio: {df_potencial['score_credito'].mean():.2f}")
print(f"Mínimo: {df_potencial['score_credito'].min():.2f}")
print(f"Máximo: {df_potencial['score_credito'].max():.2f}")

# Comparacion con el resto
print("\nPotenciales y resto (sin crédito)")

df_resto = df_bin[
    (df_bin["score_credito"] < umbral) &
    (df_bin["tiene_credito_personal"] == 0)
]

print(f"\n{'Métrica':<30} {'Potenciales':>15} {'Resto':>15} {'Diferencia':>15}")

metricas = {
    "Edad promedio": ("edad", "normal"),
    "Ingreso promedio": ("ingreso_actividad_principal", "dinero"),
    "Score promedio": ("score_credito", "normal"),
    "% Educación alta": ("educacion_alta", "porcentaje"),
    "% Buenos hábitos": ("buen_habito", "porcentaje"),
    "% Ingreso fijo": ("ingreso_fijo", "porcentaje"),
    "% Tiene nómina": ("filtro_tiene_nomina", "porcentaje")
}

for nombre, (columna, formato) in metricas.items():
    pot = df_potencial[columna].mean()
    res = df_resto[columna].mean()
    dif = pot - res

    if formato == "dinero":
        print(f"{nombre:<30} ${pot:>14,.0f} ${res:>14,.0f} ${dif:>+14,.0f}")
    elif formato == "porcentaje":
        print(f"{nombre:<30} {pot*100:>14.1f}% {res*100:>14.1f}% {dif*100:>+14.1f}%")
    else:
        print(f"{nombre:<30} {pot:>15.2f} {res:>15.2f} {dif:>+15.2f}")

#Rangos de ingreso potenciales
print("\nDistribución de rangos de ingresos (Potenciales)")

rangos_labels = ["<5k", "5-10k", "10-15k", "15-25k", "25-50k", ">50k"]
dist_ingreso = df_potencial["puntos_ingreso"].value_counts().sort_index()

for i, cuenta in enumerate(dist_ingreso):
    pct = 100 * cuenta / len(df_potencial)
    print(f"{rangos_labels[i]:<15} {cuenta:>5} ({pct:>5.1f}%)")

#Ocupaciones potenciales
print("\nTop 5 ocupaciones potenciales")
top_ocupaciones = df_potencial["ocupacion"].value_counts().head()
for i, (ocupacion, cuenta) in enumerate(top_ocupaciones.items(), 1):
    pct = 100 * cuenta / len(df_potencial)
    print(f"{i}. {ocupacion}: {cuenta:,} ({pct:.1f}%)")

#Resumen estadistico
print("\nResumen estadistico")


perfil = df_potencial[[
    "edad",
    "ingreso_actividad_principal",
    "score_credito",
    "puntos_ingreso"
]].describe()

print(perfil)


Métricas del mercado potencial

Total de personas: 13,502
Sin crédito personal: 12,904
Potenciales (buen perfil): 3,750

Representan el 27.77% del total
Representan el 29.06% de quienes no tienen crédito

Umbral del score: 5.00
Perfil del segmento potencia

Edad:
Promedio: 39.6 años
Mediana: 38 años

Ingreso:
Promedio: $16,960.51
Mediana: $6,000.00

Características:
Educación alta: 64.2%
Buenos hábitos: 50.0%
Buenos conocimientos: 70.3%
Estabilidad financiera: 30.0%
Ingreso fijo: 58.4%
Tiene nómina: 69.8%

Score:
Promedio: 7.01
Mínimo: 5.00
Máximo: 13.00

Potenciales y resto (sin crédito)

Métrica                            Potenciales           Resto      Diferencia
Edad promedio                            39.56           47.14           -7.58
Ingreso promedio               $        16,961 $         3,011 $       +13,949
Score promedio                            7.01            2.04           +4.97
% Educación alta                         64.2%            9.5%          +54.7%
% Bueno

In [ ]:
df_plot = df_bin[df_bin["tiene_credito_personal"] == 0]

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_potencial["score_credito"],
    opacity=0.7,
    name="Potenciales",
    nbinsx=20
))

fig.add_trace(go.Histogram(
    x=df_resto["score_credito"],
    opacity=0.5,
    name="Resto",
    nbinsx=20
))

fig.update_layout(
    title="Distribución del Score - Potenciales vs Resto",
    xaxis_title="Score de crédito",
    yaxis_title="Frecuencia",
    barmode="overlay",
)

fig.show()

In [ ]:
metrics = {
    "Educación alta": ("educacion_alta"),
    "Buenos hábitos": ("buen_habito"),
    "Buenos conocimientos": ("buen_conocimiento"),
    "Estabilidad financiera": ("estabilidad_financiera"),
    "Ingreso fijo": ("ingreso_fijo"),
    "Tiene nómina": ("filtro_tiene_nomina"),
}

df_compare = []

for name, col in metrics.items():
    df_compare.append({
        "Métrica": name,
        "Segmento": "Potenciales",
        "Valor": df_potencial[col].mean() * 100
    })
    df_compare.append({
        "Métrica": name,
        "Segmento": "Resto",
        "Valor": df_resto[col].mean() * 100
    })

df_compare = pd.DataFrame(df_compare)

fig = px.bar(
    df_compare,
    x="Métrica",
    y="Valor",
    color="Segmento",
    barmode="group",
    title="Comparación de indicadores clave entre segmentos",
    color_discrete_map={
        "Potenciales": "#D4A5A5",
        "Resto": "#2F3A56"
    }
)

fig.update_layout(yaxis_title="%")
fig.show()


In [ ]:

radar_labels = [
    "Educación alta",
    "Buenos hábitos",
    "Buenos conocimientos",
    "Estabilidad financiera",
    "Ingreso fijo",
    "Tiene nómina"
]

pot_values = [
    df_potencial["educacion_alta"].mean()*100,
    df_potencial["buen_habito"].mean()*100,
    df_potencial["buen_conocimiento"].mean()*100,
    df_potencial["estabilidad_financiera"].mean()*100,
    df_potencial["ingreso_fijo"].mean()*100,
    df_potencial["filtro_tiene_nomina"].mean()*100
]

rest_values = [
    df_resto["educacion_alta"].mean()*100,
    df_resto["buen_habito"].mean()*100,
    df_resto["buen_conocimiento"].mean()*100,
    df_resto["estabilidad_financiera"].mean()*100,
    df_resto["ingreso_fijo"].mean()*100,
    df_resto["filtro_tiene_nomina"].mean()*100
]

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=pot_values,
    theta=radar_labels,
    fill='toself',
    name='Potenciales',
    line=dict(color="#7B1E3A"),
    fillcolor="rgba(123,30,58,0.50)"
))

# Resto — DORADO SUAVE
fig.add_trace(go.Scatterpolar(
    r=rest_values,
    theta=radar_labels,
    fill='toself',
    name='Resto',
    line=dict(color="#F6F0E1"),
    fillcolor="rgba(226,194,117,0.40)"
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0,100])
    ),
    title="Perfil del Segmento – Radar Comparativo",
    legend=dict(title="Segmento")
)

fig.show()


In [ ]:

metrics = {
    "Educación alta": ("educacion_alta"),
    "Buenos hábitos": ("buen_habito"),
    "Buenos conocimientos": ("buen_conocimiento"),
    "Estabilidad financiera": ("estabilidad_financiera"),
    "Ingreso fijo": ("ingreso_fijo"),
    "Tiene nómina": ("filtro_tiene_nomina"),
}

df_compare = []

for name, col in metrics.items():
    df_compare.append({
        "Métrica": name,
        "Segmento": "Potenciales",
        "Valor": df_potencial[col].mean() * 100
    })
    df_compare.append({
        "Métrica": name,
        "Segmento": "Resto",
        "Valor": df_resto[col].mean() * 100
    })

df_compare = pd.DataFrame(df_compare)

fig = px.bar(
    df_compare,
    x="Métrica",
    y="Valor",
    color="Segmento",
    barmode="group",
    title="Comparación de indicadores clave entre segmentos"
)

fig.update_layout(yaxis_title="%")
fig.show()


**Resultado**

El grupo de personas con alto potencial de crédito personal representa casi un tercio de la población sin crédito y se caracteriza por una combinación de factores como la estabilidad económica, la formalidad laboral y buenos comportamientos financieros.
Aunque muchos tienen ingresos bajos o medios, presentan otras características, por ejemplo,  pagan a tiempo, llevan presupuesto, entienden conceptos financieros importantes y muestran hábitos que predicen buen desempeño crediticio.

El nivel de educación es más alto que el del resto, lo que se sugiere que tiene empleos formales con ingreso fijo y nómina lo que los convierte en un grupo con ingresos estables y predecibles, lo que reduce el riesgo crediticio.

Este grupo presenta un perfil confiable y altamente alineado con lo que una institución busca para otorgar crédito personal: estabilidad, formalidad y buenos hábitos financieros.

### Brecha de acceso a productos financieros entre poblaciones indígenas y no indígenas e indicadores sintéticos de acceso financieros

In [ ]:
import numpy as np

def binarizar(df, columnas):
    mapa = {
        'Sí': 1, 'Si': 1, 'SÍ': 1, 'SI': 1, 'sí': 1, 'si': 1,
        'No': 0, 'NO': 0, 'no': 0,
        'SA': None, 'sa': None,
        'Ns/Nc': None, 'NS/NC': None, 'ns/nc': None,
        'No sabe': None, 'No responde': None,
        'Nunca lo ha solicitado': 0 # Added this mapping
    }

    mapa_frecuencia = {
        'Siempre': 1,
        'Algunas veces': 0,
        'Nunca': 0,
        'No sabe': None,
        'No responde': None
    }

    mapa_verdadero_falso = {
        'Verdadera': 1, 'Verdadero': 1, 'verdadera': 1, 'verdadero': 1,
        'Falsa': 0, 'Falso': 0, 'falsa': 0, 'falso': 0,
        'No sabe': None,
        'No responde': None
    }

    # Specific map for 'ingreso_fijo_o_variable'
    mapa_ingreso_fijo_variable = {
        'Fijo': 1,
        'Variable': 0,
        'No sabe': None,
        'No responde': None,
        'nan': None # Handle string 'nan' after astype(str)
    }

    # Specific map for 'tiene_seguro_medico_laboral'
    mapa_seguro_medico_laboral = {
        'Del ISSSTE': 1,
        'Del seguro social (IMSS)': 1,
        'Del ISSEMYM (para empleados del gobierno del Estado de México)': 1,
        'De Pemex': 1,
        'De las fuerzas armadas o Marina': 1,
        'De alguna institución privada (seguro de gastos médicos mayores)': 1,
        'De alguna otra institución privada': 1,
        'Entonces, carece de derecho a servicios médicos': 0, # This means no insurance
        'Entonces, carece de derecho a servicios médicos por parte de su trabajo (incluye IMSS-Bienestar, antes seguro popular, instituto de salud para el bienestar)': 0, # Added this specific string
        'No': 0, # Explicit No
        'No sabe': None,
        'No responde': None,
        'nan': None,
        'Del ISSSTE estatal': 1, # Added this specific string from error
        'De PEMEX, Defensa o Marina': 1, # Added this specific string from traceback
        'De un seguro privado de gastos médicos': 1, # Added this specific string from error
        'De otra institución': 1 # Added this specific string from error
    }

    df_copy = df.copy()

    for col in columnas:
        if col not in df_copy.columns:
            continue

        df_copy[col] = df_copy[col].astype(str).str.strip()

        if col in ['paga_cuentas_a_tiempo', 'pone_metas_economicas', 'le_sobra_dinero_fin_mes']:
            mapeo_actual = mapa_frecuencia
        elif col in ['entiende_inflacion']:
            mapeo_actual = mapa_verdadero_falso
        elif col == 'ingreso_fijo_o_variable': # New condition for the problematic column
            mapeo_actual = mapa_ingreso_fijo_variable
        elif col == 'tiene_seguro_medico_laboral': # New condition for the problematic column
            mapeo_actual = mapa_seguro_medico_laboral
        else:
            mapeo_actual = mapa

        df_copy[col] = df_copy[col].replace(mapeo_actual)
        df_copy[col] = df_copy[col].replace('nan', np.nan) # Ensure 'nan' string is handled before fillna
        df_copy[col] = df_copy[col].fillna(0).astype('int8')

    return df_copy

variables_credito = [
    "tiene_credito_nomina",
    "tiene_credito_vivienda",
    "tiene_credito_personal",
    "tiene_tarjeta_credito_bancaria",
    "tiene_tarjeta_credito_departamental",
    "le_rechazaron_credito"
]

# Binarizar columnas relevantes
df_temp_binarized = binarizar(df.copy(), variables_credito)

# Se usa df_temp_binarized para las columnas binarizadas y se_considera_indigena de df
resultado_comparacion = df_temp_binarized.groupby(df['se_considera_indigena'])[variables_credito].mean().reset_index()

resultado_comparacion[variables_credito] *= 100

resultado_comparacion

,se_considera_indigena,tiene_credito_nomina,tiene_credito_vivienda,tiene_credito_personal,tiene_tarjeta_credito_bancaria,tiene_tarjeta_credito_departamental,le_rechazaron_credito
0,No,3.721273,8.075058,4.395952,19.091292,25.089606,19.186169
1,No sabe,1.895735,2.843602,1.895735,9.004739,15.639810,13.270142
2,Sí,3.180026,5.650460,4.651774,9.618922,18.318003,16.162943


In [ ]:
resultado_filtrado = resultado_comparacion[
    resultado_comparacion['se_considera_indigena'].isin(['Sí', 'No'])
].copy()

In [ ]:
resultado_filtrado['se_considera_indigena'] = pd.Categorical(
    resultado_filtrado['se_considera_indigena'],
    categories=['No', 'Sí'],
    ordered=True
)
resultado_filtrado = resultado_filtrado.sort_values('se_considera_indigena')

In [ ]:
def crear_barras_divergentes_brecha(resultado_comparacion):

    resultado = resultado_comparacion[
        resultado_comparacion['se_considera_indigena'].isin(['Sí', 'No'])
    ].copy()

    resultado['se_considera_indigena'] = pd.Categorical(
        resultado['se_considera_indigena'],
        categories=['No', 'Sí'],
        ordered=True
    )
    resultado = resultado.sort_values('se_considera_indigena')

    # Extraer filas
    no_indigena = resultado[resultado['se_considera_indigena'] == 'No'].iloc[0]
    indigena = resultado[resultado['se_considera_indigena'] == 'Sí'].iloc[0]

    variables_credito = [
        'tiene_credito_nomina',
        'tiene_credito_vivienda',
        'tiene_credito_personal',
        'tiene_tarjeta_credito_bancaria',
        'tiene_tarjeta_credito_departamental',
        'le_rechazaron_credito'
    ]

    # Calcular brechas
    brechas = []
    for var in variables_credito:
        brecha = no_indigena[var] - indigena[var]
        brechas.append({
            'variable': var,
            'brecha': brecha,
            'favorece': 'No Indígena' if brecha > 0 else 'Indígena'
        })

    df_brechas = pd.DataFrame(brechas)

    nombres_limpios = {
        'tiene_credito_nomina': 'Tiene Crédito Nómina',
        'tiene_credito_vivienda': 'Tiene Crédito Vivienda',
        'tiene_credito_personal': 'Tiene Crédito Personal',
        'tiene_tarjeta_credito_bancaria': 'Tiene Tarjeta Bancaria',
        'tiene_tarjeta_credito_departamental': 'Tiene Tarjeta Departamental',
        'le_rechazaron_credito': 'Le han Rechazado Crédito'
    }

    df_brechas['variable'] = df_brechas['variable'].map(nombres_limpios)
    df_brechas = df_brechas.sort_values('brecha')

    colores = ['#10b981' if x < 0 else '#ef4444' for x in df_brechas['brecha']]

    fig = go.Figure()

    fig.add_trace(go.Bar(
        y=df_brechas['variable'],
        x=df_brechas['brecha'],
        orientation='h',
        marker=dict(color=colores, line=dict(color='white', width=1)),
        text=[f"{abs(x):.1f}pp" for x in df_brechas['brecha']],
        textposition='outside'
    ))

    fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=2)

    fig.update_layout(
        title='Brecha de Acceso a Productos Financieros<br><sub>Diferencia: Población Indígena - Población No Indígena</sub>',
        xaxis_title='Brecha (puntos porcentuales)',
        showlegend=False,
        height=500
    )

    return fig

def crear_indice_inclusion(resultado_comparacion):

    resultado = resultado_comparacion[
        resultado_comparacion['se_considera_indigena'].isin(['Sí', 'No'])
    ].copy()

    resultado['se_considera_indigena'] = pd.Categorical(
        resultado['se_considera_indigena'],
        categories=['No', 'Sí'],
        ordered=True
    )
    resultado = resultado.sort_values('se_considera_indigena')

    variables_positivas = [
        'tiene_credito_nomina',
        'tiene_credito_vivienda',
        'tiene_credito_personal',
        'tiene_tarjeta_credito_bancaria',
        'tiene_tarjeta_credito_departamental'
    ]

    indices = []
    for _, row in resultado.iterrows():
        tipo = 'Población Indígena' if row['se_considera_indigena'] == 'Sí' else 'Población No Indígena'
        indice = row[variables_positivas].mean()
        rechazo = row['le_rechazaron_credito']

        indices.append({
            'Población': tipo,
            'Índice de Inclusión': indice,
            'Tasa de Rechazo': rechazo
        })

    df_indices = pd.DataFrame(indices)

    colores = df_indices['Población'].map({
        'Población No Indígena': '#f59e0b',
        'Población Indígena': '#3b82f6'
    })

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Índice de Inclusión Financiera', 'Tasa de Rechazo de Crédito'),
        specs=[[{"type": "bar"}, {"type": "bar"}]]
    )

    fig.add_trace(
        go.Bar(
            x=df_indices['Población'],
            y=df_indices['Índice de Inclusión'],
            marker_color=colores,
            text=[f"{x:.1f}%" for x in df_indices['Índice de Inclusión']],
            textposition='outside'
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=df_indices['Población'],
            y=df_indices['Tasa de Rechazo'],
            marker_color=colores,
            text=[f"{x:.1f}%" for x in df_indices['Tasa de Rechazo']],
            textposition='outside'
        ),
        row=1, col=2
    )

    fig.update_layout(
        title='Indicadores Sintéticos de Acceso Financiero',
        height=500
    )

    return fig

def dashboard_indigena_credito(resultado_comparacion):

    fig2 = crear_barras_divergentes_brecha(resultado_comparacion)
    fig2.show()

    fig6 = crear_indice_inclusion(resultado_comparacion)
    fig6.show()

dashboard_indigena_credito(resultado_comparacion)

###Resultados:
La primer gráfica nos permite observar que existen diferencias significativas en el acceso a productos financieros entre ambas poblaciones. Particularmente, podemos notar que la población indígena presenta tasas muy ligeramente superiores en el acceso al crédito personal y en el rechazo de solicitudes de crédito, esto podría indicar el por qué tienen más barreras para adquirir productos más regulados como un crédito de vivienda o departamental.

Por otro lado, la segunda visualización confirma la desigualdad entre ambas posibles, indicando que la población no indígena presenta un nivel más alto de inclusión, caso contrario con las personas de origen indígena. A pesar de ello, la tasa de rechazo de crédito es bastante similar entre ambos grupos, aunque ligeramente mayor en la población indígena.

En conjunto, estas métricas reflejan una brecha estructural que limita la integración financiera plena de la población indígena.

### Tendencias y patrones entre compradores y no compradores de criptomonedas

In [ ]:
df['compro_criptomonedas'].value_counts()
df['compro_criptomonedas'].value_counts(normalize=True) * 100

,proportion
compro_criptomonedas,
No,97.978077
Sí,2.021923


In [ ]:
import numpy as np

# Redefine la función binarizar localmente para incluir 'No sabe' en el mapeo
def binarizar(df, columnas):
    mapa = {
        'Sí': 1, 'Si': 1, 'SÍ': 1, 'SI': 1, 'sí': 1, 'si': 1,
        'No': 0, 'NO': 0, 'no': 0,
        'SA': None, 'sa': None,
        'Ns/Nc': None, 'NS/NC': None, 'ns/nc': None,
        'No sabe': None, 'No responde': None # Añadido 'No sabe' y 'No responde'
    }

    mapa_frecuencia = {
        'Siempre': 1,
        'Algunas veces': 0,
        'Nunca': 0,
        'No sabe': None,
        'No responde': None
    }

    mapa_verdadero_falso = {
        'Verdadera': 1, 'Verdadero': 1, 'verdadera': 1, 'verdadero': 1,
        'Falsa': 0, 'Falso': 0, 'falsa': 0, 'falso': 0,
        'No sabe': None,
        'No responde': None
    }

    # Specific map for 'ingreso_fijo_o_variable'
    mapa_ingreso_fijo_variable = {
        'Fijo': 1,
        'Variable': 0,
        'No sabe': None,
        'No responde': None,
        'nan': None # Handle string 'nan' after astype(str)
    }

    # Specific map for 'tiene_seguro_medico_laboral'
    mapa_seguro_medico_laboral = {
        'Del ISSSTE': 1,
        'Del seguro social (IMSS)': 1,
        'Del ISSEMYM (para empleados del gobierno del Estado de México)': 1,
        'De Pemex': 1,
        'De las fuerzas armadas o Marina': 1,
        'De alguna institución privada (seguro de gastos médicos mayores)': 1,
        'De alguna otra institución privada': 1,
        'Entonces, carece de derecho a servicios médicos': 0, # This means no insurance
        'Entonces, carece de derecho a servicios médicos por parte de su trabajo (incluye IMSS-Bienestar, antes seguro popular, instituto de salud para el bienestar)': 0, # Added this specific string
        'No': 0, # Explicit No
        'No sabe': None,
        'No responde': None,
        'nan': None,
        'Del ISSSTE estatal': 1, # Added this specific string from error
        'De PEMEX, Defensa o Marina': 1, # Added this specific string from traceback
        'De un seguro privado de gastos médicos': 1, # Added this specific string from error
        'De otra institución': 1 # Added this specific string from error
    }

    df_copy = df.copy()

    for col in columnas:
        if col not in df_copy.columns:
            continue

        df_copy[col] = df_copy[col].astype(str).str.strip()

        if col in ['paga_cuentas_a_tiempo', 'pone_metas_economicas', 'le_sobra_dinero_fin_mes']:
            mapeo_actual = mapa_frecuencia
        elif col in ['entiende_inflacion']:
            mapeo_actual = mapa_verdadero_falso
        elif col == 'ingreso_fijo_o_variable': # New condition for the problematic column
            mapeo_actual = mapa_ingreso_fijo_variable
        elif col == 'tiene_seguro_medico_laboral': # New condition for the problematic column
            mapeo_actual = mapa_seguro_medico_laboral
        else:
            mapeo_actual = mapa

        df_copy[col] = df_copy[col].replace(mapeo_actual)
        df_copy[col] = df_copy[col].replace('nan', np.nan) # Ensure 'nan' string is handled before fillna
        df_copy[col] = df_copy[col].fillna(0).astype('int8')

    return df_copy

# Binarizar las columnas necesarias antes de la agrupación
columns_to_binarize = [
    "compro_criptomonedas",
    "tiene_credito_personal",
    "tiene_credito_nomina"
]
df_limp_temp = binarizar(df.copy(), columns_to_binarize)

# Ahora perform the groupby and mean calculation
result = df_limp_temp.groupby("compro_criptomonedas")[
    ["tiene_credito_personal", "tiene_credito_nomina"]
].mean() * 100

print(result)


                      tiene_credito_personal  tiene_credito_nomina
compro_criptomonedas                                              
0                                   4.240683              3.477209
1                                  13.553114              6.593407


In [ ]:
df_temp = df_limp.copy()
df_temp = binarizar(df_temp, ["tiene_cuenta_nomina"])

df_temp.groupby("compro_criptomonedas")[
    ["tiene_credito_nomina", "tiene_cuenta_nomina",
     "ingreso_fijo_o_variable"]
].mean() * 100

,tiene_credito_nomina,tiene_cuenta_nomina,ingreso_fijo_o_variable
compro_criptomonedas,,,
No,3.477209,31.317560,29.866203
Sí,6.593407,59.340659,41.025641


In [ ]:
df.groupby("compro_criptomonedas")["edad"].mean()

,edad
compro_criptomonedas,
No,45.028347
Sí,35.201465


In [ ]:
df.groupby("compro_criptomonedas")["nivel_educativo"].value_counts(normalize=True) * 100

compro_criptomonedas  nivel_educativo                             
No                    Secundaria                                      27.197823
                      Preparatoria o bachillerato                     20.810341
                      Licenciatura o ingeniería (profesional)         20.455061
                      Primaria                                        20.379469
                      Ninguno                                          4.021468
                      Estudios técnicos con preparatoria terminada     2.403810
                      Estudios técnicos con secundaria terminada       1.965379
                      Maestría                                         1.783959
                      Especialidad                                     0.370398
                      Doctorado                                        0.309925
                      Preescolar o kínder                              0.151183
                      Normal básica                                    0.113387
                      No sabe                                          0.037796
Sí                    Licenciatura o ingeniería (profesional)         53.846154
                      Preparatoria o bachillerato                     20.146520
                      Secundaria                                      13.553114
                      Maestría                                         7.326007
                      Estudios técnicos con preparatoria terminada     2.197802
                      Doctorado                                        1.098901
                      Especialidad                                     1.098901
                      Estudios técnicos con secundaria terminada       0.366300
                      Primaria                                         0.366300
Name: proportion, dtype: float64

In [ ]:
df.groupby("compro_criptomonedas")[
    ["ingreso_actividad_principal"]
].mean()

,ingreso_actividad_principal
compro_criptomonedas,
No,10002.238732
Sí,22012.526104


In [ ]:
columnas_a_binarizar_para_confianza = [
    "compro_criptomonedas",
    "confia_seguridad_dinero",
    "confia_proteger_datos",
    "confia_resolver_quejas",
    "confia_resolver_necesidades"
]

df_confianza_cripto = binarizar(df.copy(), columnas_a_binarizar_para_confianza)

df_confianza_cripto.groupby("compro_criptomonedas")[
    ["confia_seguridad_dinero", "confia_proteger_datos",
     "confia_resolver_quejas", "confia_resolver_necesidades"]
].mean() * 100

,confia_seguridad_dinero,confia_proteger_datos,confia_resolver_quejas,confia_resolver_necesidades
compro_criptomonedas,,,,
0,61.289591,60.072568,54.902109,54.917227
1,72.527473,62.637363,64.102564,66.300366


In [ ]:
from plotly.subplots import make_subplots
import pandas as pd
import plotly.express as px # Ensure plotly.express is imported
import plotly.graph_objects as go # Ensure plotly.graph_objects is imported
import numpy as np # Ensure numpy is imported for np.nan

# Re-defining binarizar to ensure it's available and includes all necessary mappings.
# Taking the latest robust binarizar function from cell Y5FElIFHqRT3
def binarizar(df, columnas):
    mapa = {
        'Sí': 1, 'Si': 1, 'SÍ': 1, 'SI': 1, 'sí': 1, 'si': 1,
        'No': 0, 'NO': 0, 'no': 0,
        'SA': None, 'sa': None,
        'Ns/Nc': None, 'NS/NC': None, 'ns/nc': None,
        'No sabe': None, 'No responde': None,
        'Nunca lo ha solicitado': 0 # Added this mapping
    }

    mapa_frecuencia = {
        'Siempre': 1,
        'Algunas veces': 0,
        'Nunca': 0,
        'No sabe': None,
        'No responde': None
    }

    mapa_verdadero_falso = {
        'Verdadera': 1, 'Verdadero': 1, 'verdadera': 1, 'verdadero': 1,
        'Falsa': 0, 'Falso': 0, 'falsa': 0, 'falso': 0,
        'No sabe': None,
        'No responde': None
    }

    # Specific map for 'ingreso_fijo_o_variable'
    mapa_ingreso_fijo_variable = {
        'Fijo': 1,
        'Variable': 0,
        'No sabe': None,
        'No responde': None,
        'nan': None # Handle string 'nan' after astype(str)
    }

    # Specific map for 'tiene_seguro_medico_laboral'
    mapa_seguro_medico_laboral = {
        'Del ISSSTE': 1,
        'Del seguro social (IMSS)': 1,
        'Del ISSEMYM (para empleados del gobierno del Estado de México)': 1,
        'De Pemex': 1,
        'De las fuerzas armadas o Marina': 1,
        'De alguna institución privada (seguro de gastos médicos mayores)': 1,
        'De alguna otra institución privada': 1,
        'Entonces, carece de derecho a servicios médicos': 0, # This means no insurance
        'Entonces, carece de derecho a servicios médicos por parte de su trabajo (incluye IMSS-Bienestar, antes seguro popular, instituto de salud para el bienestar)': 0, # Added this specific string
        'No': 0, # Explicit No
        'No sabe': None,
        'No responde': None,
        'nan': None,
        'Del ISSSTE estatal': 1, # Added this specific string from error
        'De PEMEX, Defensa o Marina': 1, # Added this specific string from traceback
        'De un seguro privado de gastos médicos': 1, # Added this specific string from error
        'De otra institución': 1 # Added this specific string from error
    }

    df_copy = df.copy()

    for col in columnas:
        if col not in df_copy.columns:
            continue

        df_copy[col] = df_copy[col].astype(str).str.strip()

        if col in ['paga_cuentas_a_tiempo', 'pone_metas_economicas', 'le_sobra_dinero_fin_mes']:
            mapeo_actual = mapa_frecuencia
        elif col in ['entiende_inflacion']:
            mapeo_actual = mapa_verdadero_falso
        elif col == 'ingreso_fijo_o_variable': # New condition for the problematic column
            mapeo_actual = mapa_ingreso_fijo_variable
        elif col == 'tiene_seguro_medico_laboral': # New condition for the problematic column
            mapeo_actual = mapa_seguro_medico_laboral
        else:
            mapeo_actual = mapa

        df_copy[col] = df_copy[col].replace(mapeo_actual)
        df_copy[col] = df_copy[col].replace('nan', np.nan) # Ensure 'nan' string is handled before fillna
        df_copy[col] = df_copy[col].fillna(0).astype('int8')

    return df_copy


def crear_donut_compradores(df):
    """
    Gráfico de dona mostrando proporción de compradores
    """
    counts = df['compro_criptomonedas'].value_counts()
    percentages = df['compro_criptomonedas'].value_counts(normalize=True) * 100

    labels = counts.index.tolist()
    values = counts.values.tolist()
    texts = [f"{pct:.1f}%" for pct in percentages.values]

    colors = ['#10b981' if label == 'Sí' else '#ef4444' for label in labels]

    fig = go.Figure(data=[go.Pie(
        labels=labels,
        values=values,
        hole=0.5,
        marker=dict(colors=colors, line=dict(color='white', width=2)),
        text=texts,
        textinfo='label+text',
        textfont=dict(size=14),
        hovertemplate='%{label}<br>%{value:,} personas<br>%{percent}<extra></extra>'
    )])

    fig.update_layout(
        title='Distribución de Compradores de Criptomonedas',
        annotations=[dict(text=f'Total<br>{sum(values):,}', x=0.5, y=0.5,
                          font_size=16, showarrow=False)],
        height=400,
        showlegend=True
    )

    return fig


def crear_barras_productos_financieros(df_limp_original):
    """
    Comparación de tenencia de productos financieros
    """
    df_limp = df_limp_original.copy()

    columns_for_this_plot = ["compro_criptomonedas", "tiene_credito_personal", "tiene_credito_nomina"]

    df_limp = binarizar(df_limp, columns_for_this_plot)

    productos = ["tiene_credito_personal", "tiene_credito_nomina"]
    resultado = df_limp.groupby("compro_criptomonedas")[productos].mean() * 100

    df_melted = resultado.reset_index().melt(
        id_vars='compro_criptomonedas',
        var_name='Producto',
        value_name='Porcentaje'
    )

    df_melted['Producto'] = df_melted['Producto'].map({
        'tiene_credito_personal': 'Crédito Personal',
        'tiene_credito_nomina': 'Crédito Nómina'
    })

    df_melted['compro_criptomonedas'] = df_melted['compro_criptomonedas'].map({
        1: 'Compra Cripto',
        0: 'No Compra Cripto'
    })

    fig = px.bar(
        df_melted,
        x='Porcentaje',
        y='Producto',
        color='compro_criptomonedas',
        orientation='h',
        barmode='group',
        color_discrete_map={
            'Compra Cripto': '#10b981',
            'No Compra Cripto': '#6b7280'
        },
        text='Porcentaje',
        title='Tenencia de Productos Financieros'
    )

    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(
        xaxis_title='Porcentaje (%)',
        yaxis_title='',
        legend_title='',
        height=350
    )

    return fig

def crear_boxplot_edad(df_original):
    """
    Box plot comparando distribución de edad
    """
    df = df_original.copy()
    df = binarizar(df, ['compro_criptomonedas']) # Binarize for consistency

    fig = go.Figure()

    # Map binarized values back to strings for display
    df['compro_criptomonedas_label'] = df['compro_criptomonedas'].map({1: 'Sí', 0: 'No'})

    for categoria in df['compro_criptomonedas_label'].unique():
        datos = df[df['compro_criptomonedas_label'] == categoria]['edad'].dropna()
        nombre = 'Compra Cripto' if categoria == 'Sí' else 'No Compra Cripto'
        color = '#10b981' if categoria == 'Sí' else '#6b7280'

        fig.add_trace(go.Box(
            y=datos,
            name=nombre,
            marker_color=color,
            boxmean='sd',
            hovertemplate='%{y} años<extra></extra>'
        ))

    # Agregar línea de promedio
    edad_prom_si = df[df['compro_criptomonedas_label'] == 'Sí']['edad'].mean()
    edad_prom_no = df[df['compro_criptomonedas_label'] == 'No']['edad'].mean()

    fig.update_layout(
        title=f'Distribución de Edad por Grupo<br><sub>Promedio Cripto: {edad_prom_si:.1f} años | No Cripto: {edad_prom_no:.1f} años</sub>',
        yaxis_title='Edad (años)',
        showlegend=True,
        height=450
    )

    return fig


def crear_violin_ingresos(df_original):
    """
    Violin plot para distribución de ingresos
    """
    df = df_original.copy()
    df = binarizar(df, ['compro_criptomonedas']) # Binarize for consistency

    df_filtered = df[df['ingreso_actividad_principal'] < df['ingreso_actividad_principal'].quantile(0.99)]

    fig = go.Figure()

    df_filtered['compro_criptomonedas_label'] = df_filtered['compro_criptomonedas'].map({1: 'Sí', 0: 'No'})

    for categoria in df_filtered['compro_criptomonedas_label'].unique():
        datos = df_filtered[df_filtered['compro_criptomonedas_label'] == categoria]['ingreso_actividad_principal'].dropna()
        nombre = 'Compra Cripto' if categoria == 'Sí' else 'No Compra Cripto'
        color = '#10b981' if categoria == 'Sí' else '#6b7280'

        fig.add_trace(go.Violin(
            y=datos,
            name=nombre,
            box_visible=True,
            meanline_visible=True,
            fillcolor=color,
            opacity=0.6,
            line_color=color,
            hovertemplate='$%{y:,.0f}<extra></extra>'
        ))

    fig.update_layout(
        title='Distribución de Ingresos por Actividad Principal',
        yaxis_title='Ingreso ($)',
        showlegend=True,
        height=450,
        yaxis_tickformat='$,.0f'
    )

    return fig


def crear_dashboard_criptomonedas(df, df_limp_arg, df_temp, df_confianza_cripto):

    # Ensure df_limp_arg contains binarized 'compro_criptomonedas' and 'tiene_credito_personal'
    # The previous cell where df_limp was created did not include 'compro_criptomonedas' or 'tiene_credito_personal'
    # in its binarization list. So, we re-binarize here for this function's scope.
    df_limp_for_plot = binarizar(df_limp_arg.copy(), ["compro_criptomonedas", "tiene_credito_personal"])


    print("Proporción de Compradores.")
    fig1 = crear_donut_compradores(df) # This uses the original df, which is fine for value_counts()
    fig1.show()

    print(" Productos Financieros.")
    fig2 = crear_barras_productos_financieros(df_limp_for_plot) # Pass the correctly binarized df_limp
    fig2.show()


    print("Distribución de Edad.")
    fig3 = crear_boxplot_edad(df) # This uses the original df and binarizes internally
    fig3.show()


    print(" Distribución de sueldo.")
    fig = crear_violin_ingresos(df) # This uses the original df and binarizes internally
    fig.show()


crear_dashboard_criptomonedas(df, df_limp, df_temp, df_confianza_cripto)

Proporción de Compradores.


 Productos Financieros.


Distribución de Edad.


 Distribución de sueldo.


In [ ]:
df.to_csv("df.csv", index=False, encoding="utf-8")


In [ ]:
df.to_csv("/content/drive/MyDrive/df.csv", index=False)


In [ ]:
df.to_csv('/content/drive/MyDrive/Proyecto_final_DS/df_saved_to_drive.csv', index=False)
print("DataFrame 'df' guardado exitosamente en Google Drive como 'df_saved_to_drive.csv'")

DataFrame 'df' guardado exitosamente en Google Drive como 'df_saved_to_drive.csv'


###Resultado:
Estas gráficas demuestran que la adopción de criptomonedas es aún muy limitada en la población analizada, teniendo una población muy reducida de apenas el 2%. Sin embargo, este grupo presenta patrones financieros y sociodemográficos claramente diferenciados respecto al resto de personas.

En términos financieros, las personas que compran criptomonedas tiene una mayor participación en tanto en crédito personal como en nómina, lo que nos da a entender que son usuarios más activos en el sistema financiero y con mayor disposición a productos crediticios. Esta tendencia coincide con señales de mayor estabilidad o vinculación laboral.

Por último, los compradores de criptomonedas son notablemente más jovenes y presentan una distribución educativa más elevada, lo que indica que la compra de criptomonedas está asociada a perfiles con mayor escolaridad.

##Conclusión final:
Este análisis nos permite identificar algunos patrones interesantes sobre el acceso al crédito, el comportamiento y cultura financiera, y la confianza que tienen las personas a las instituciones bancarias.

Durante la elaboración de este proyecto, pudimos observar diferencias relevantes en la tenencia de productos financieros. El crédito de nómia, principalmente asociado al mundo laboral y formalidad, presenta mayores tasas de uso entre personas con ingresos estables y relación con el banco. Por otra parte, el crédito personal apareció como un producto más accesible para perfiles con menor estabilidad. Esta diferencia nos puede dar a entender que la estructura del sector crediticio sigue generando brechas basadas en la situación laboral de cada persona.

Pudimos encontrar que en cuanto a comportamiento crediticio, los atrasos, rechazos de crédito y evaluación de deuda total reflejan diferencias significativas en el score implícito que puede tener cada persona. Los niveles más altos de rechazo suelen coincidir con menor acceso al crédito formal, esto nos confirma que las barreras de entrada persisten para sectores con vulnerabilidad financiera.

De igual forma, los indicadores de confianza en los bancos nos demuestran que una parte considerable de la población mantiene dudas sobre la seguridad de su dinero o la protección de sus datos. Esto podría influir en la decisión de utilizar o evitar productos financieros formales, afectando indirectamente la inclusión financiera.

Para robustecer este análisis, se analizaron los porcentajes de inclusión y accesos a diferentes tipos de créditos entre población indígena y personas que no se consideran. Los resultados obtenidos nos demuestran que las personas de origenes indígenas aún tienen una barrera amplia de acceso a créditos de nómina y viviendas, principalmente, además de presentar un porcentaje más amplio de rechazos de los mismos. Esto revela que esta población minoritaria presenta limitaciones en cuando a adquisicion de créditos.

Finalmente, se analizaron los tipos de créditos y perfiles financieros de personas que adquieren criptomonedas contra las que no lo hacem. Los resultados nos revelaron que este grupo reducido de personas tienen niveles de estudios más altos y acceso a créditos personales y de nómina superiores. Lo que nos da a entender que estos compradores conforman un grupo más joven, educados y financieramente activos, esta caracterización permite comprender su comportamiento y abre oportunidades para explorar estrategias de inclusión financiera orientadas a perfiles emrgentes en el ecosistema digital.

